# Notebook 10 -- Benchmark Comparativo: MoE vs. Modelos Convencionales

---


## Objetivo

Torneo masivo: 30 modelos (3 familias × 10) vs. MoE. Análisis 4D: desempeño predictivo, eficiencia computacional, interpretabilidad (SHAP + routing), robustez (stress test COVID).


## Entrada

features + modelo MoE


## Salida

Tabla maestra (MoE vs. 30 modelos) + ranking por métrica + análisis de interpretabilidad + reporte de robustez


## Instalación de dependencias

In [ ]:
# Instalación de dependencias
# En Docker (make up / make pipeline) ya vienen instaladas con versiones fijas
# (requirements.txt). En Google Colab, después de clonar el repo y hacer
# %cd notebooks, se instalan las mismas versiones:
import sys
if "google.colab" in sys.modules:
    !pip install -q -r ../requirements.txt

# Verificar PyTorch
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Configuración e Imports

In [ ]:
# ============================================================
# CONFIGURACIÓN — Modificar según entorno
# ============================================================
DATA_DIR    = "../data"
MODELS_DIR  = "../models"
RESULTS_DIR = "../results"
# Figuras en inglés (versión para publicación); la carpeta no se versiona
os.makedirs(f"{RESULTS_DIR}/english", exist_ok=True)
PROD_DIR    = f"{MODELS_DIR}/production"
RANDOM_SEED = 42
N_JOBS      = -1

# Bergstra & Bengio (2012): n=60 → P(top 5%) ≥ 0.95
N_RANDOM_SEARCH_ITER = 60
N_CV_FOLDS = 3  # TimeSeriesSplit folds

import os, json, pickle, time, warnings, gc, copy
from datetime import datetime, timedelta
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats as sp_stats
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             mean_absolute_percentage_error, r2_score)
from sklearn.experimental import enable_halving_search_cv # Added this line
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, HalvingRandomSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline

# ML models
from sklearn.linear_model import (Ridge, Lasso, ElasticNet, BayesianRidge,
                                  HuberRegressor, SGDRegressor)
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              ExtraTreesRegressor, AdaBoostRegressor,
                              BaggingRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import lightgbm as lgb
import xgboost as xgb

# Statistical models
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.ar_model import AutoReg

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.fontsize': 9, 'axes.grid': True, 'grid.alpha': 0.3,
})

# Directorios de resultados del benchmark
BENCH_DIR = Path(RESULTS_DIR) / 'benchmark'
BENCH_DIR.mkdir(parents=True, exist_ok=True)
print(f"Directorio de benchmark: {BENCH_DIR}")

---
## 1. Carga de Datos y Preparación

Se carga el mismo `features.parquet` utilizado en todo el pipeline MoE para garantizar comparabilidad.

In [ ]:
# ============================================================
# CARGA DE DATOS
# ============================================================
features_path = Path(DATA_DIR) / 'processed' / 'features.parquet'

if features_path.exists():
    df = pd.read_parquet(features_path)
    if not isinstance(df.index, pd.DatetimeIndex):
        for col in ['timestamp', 'datetime', 'date']:
            if col in df.columns:
                df = df.set_index(col)
                break
    df = df.sort_index()
    print(f"✓ Datos cargados: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"  Rango: {df.index.min()} → {df.index.max()}")
else:
    # Sin fallback a datos sintéticos: el benchmark debe correr sobre los datos reales
    raise FileNotFoundError(
        f"No existe {features_path}. Ejecuta antes los notebooks 01 y 02 (make pipeline)."
    )

# Target y features
calendar_only = [
    'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'month_sin', 'month_cos',
    'is_holiday', 'is_bridge', 'days_to_next_holiday', 'days_from_prev_holiday',
]

all_cols = list(df.columns)

# Lag features
lag_features = sorted([c for c in all_cols if c.startswith('demand_lag_')])

calendar_lags = calendar_only + lag_features

TARGET = 'demand'
EXCLUDE_COLS = [TARGET, 'type_day', 'type_week', 'cluster']
#feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS
#                and not c.startswith('type_') and df[c].dtype in ['float64','float32','int64','int32']]

feature_cols = calendar_lags

print(f"\n  Target: {TARGET}")
print(f"  Features ({len(feature_cols)}): {feature_cols[:10]}...")

# Eliminar filas con NaN
df_clean = df[[TARGET] + feature_cols].dropna()
print(f"  Filas limpias: {df_clean.shape[0]:,}")

In [ ]:
df_clean.head()

In [ ]:
df_clean = df_clean[~df_clean.index.year.isin([2020, 2021])]

In [ ]:
df_clean.head()

In [ ]:
df_clean.shape

### 1.1 División temporal Train / Validation / Test

Se usa la misma partición temporal del pipeline MoE:
- **Train**: 70%
- **Validation**: 15%
- **Test**: 15%

In [ ]:
# ============================================================
# SPLIT TEMPORAL
# ============================================================
n_total = len(df_clean)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)
n_test  = n_total - n_train - n_val

df_train = df_clean.iloc[:n_train]
df_val   = df_clean.iloc[n_train:n_train + n_val]
df_test  = df_clean.iloc[n_train + n_val:]

X_train, y_train = df_train[feature_cols], df_train[TARGET]
X_val,   y_val   = df_val[feature_cols],   df_val[TARGET]
X_test,  y_test  = df_test[feature_cols],  df_test[TARGET]

# Combinado train+val para HPO con TimeSeriesSplit
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

# Serie temporal para modelos estadísticos puros
ts_train = y_train.copy()
ts_val   = y_val.copy()
ts_test  = y_test.copy()

# Scaler para DL
scaler_X = StandardScaler().fit(X_train)
scaler_y = StandardScaler().fit(y_train.values.reshape(-1, 1))

print(f"Train: {len(df_train):,} ({df_train.index.min().date()} → {df_train.index.max().date()})")
print(f"Val:   {len(df_val):,}  ({df_val.index.min().date()} → {df_val.index.max().date()})")
print(f"Test:  {len(df_test):,}  ({df_test.index.min().date()} → {df_test.index.max().date()})")

### 1.2 Funciones auxiliares de evaluación

In [ ]:
# ============================================================
# MÉTRICAS Y UTILIDADES
# ============================================================
def compute_metrics(y_true, y_pred, prefix=''):
    """Calcula métricas estándar de regresión."""
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {
        f'{prefix}MAE':  mean_absolute_error(y_true, y_pred),
        f'{prefix}RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        f'{prefix}MAPE': mape,
        f'{prefix}R2':   r2_score(y_true, y_pred),
    }

def timer_context():
    """Context manager para medir tiempos."""
    class Timer:
        def __enter__(self):
            self.start = time.perf_counter()
            return self
        def __exit__(self, *args):
            self.elapsed = time.perf_counter() - self.start
    return Timer()

def print_screening_results(results_dict, family_name):
    """Imprime tabla resumen de screening."""
    rows = []
    for name, r in results_dict.items():
        rows.append({
            'Modelo': name,
            'MAPE_val (%)': r['val_metrics']['MAPE'],
            'RMSE_val': r['val_metrics']['RMSE'],
            'MAE_val': r['val_metrics']['MAE'],
            'R2_val': r['val_metrics']['R2'],
            'Tiempo_train (s)': r['train_time'],
        })
    res_df = pd.DataFrame(rows).sort_values('MAPE_val (%)')
    print(f"\n{'='*80}")
    print(f"  SCREENING — Familia {family_name} ({len(res_df)} modelos)")
    print(f"{'='*80}")
    print(res_df.to_string(index=False, float_format='{:.4f}'.format))
    return res_df

print("✓ Funciones auxiliares definidas")

---
## 2. Carga del Modelo MoE (Baseline)

Se carga el modelo MoE serializado en `models/production/moe_forecaster.pkl` para obtener sus predicciones en el test set como referencia.

In [ ]:
# ============================================================
# CARGAR MODELO MoE
# ============================================================
# La clase vive en src/moe_forecaster/forecaster.py para que moe_forecaster.pkl
# se pueda cargar fuera de este notebook: si se define aquí, pickle la guarda como
# __main__.MoEForecaster y solo existe dentro de esta sesión de Jupyter.
import os, sys
sys.path.insert(0, os.path.abspath("../src"))
from moe_forecaster import MoEForecaster
print('Clase MoEForecaster importada desde src/moe_forecaster.')

In [ ]:
# Now, attempt to load the MoE model again with the placeholder class defined
moe_path = Path(PROD_DIR) / 'moe_forecaster.pkl'

if moe_path.exists():
    try:
        with open(moe_path, 'rb') as f:
            moe_model = pickle.load(f)
        print("✓ Modelo MoE cargado desde producción")

        # Inspect the loaded object
        print(f"\nTipo de objeto cargado: {type(moe_model)}")
        print("Atributos del modelo (ejemplos):")
        for attr in ['num_experts', 'input_dim', 'experts', 'gating_network', 'scalers']:
            if hasattr(moe_model, attr):
                print(f"  - {attr}: {getattr(moe_model, attr)}")

        # Predicciones MoE en test
        with timer_context() as t_moe_infer:
            moe_preds_test = moe_model.predict(df_test)

        moe_metrics_test = compute_metrics(y_test, moe_preds_test)
        moe_train_time = 0  # Ya entrenado, se documenta aparte
        moe_infer_time = t_moe_infer.elapsed

        print(f"\n  MoE Test — MAPE: {moe_metrics_test['MAPE']:.4f}% | RMSE: {moe_metrics_test['RMSE']:.2f}")
        print(f"  Tiempo inferencia test: {moe_infer_time:.3f}s")
    except Exception as e:
        # El benchmark no tiene sentido sin el modelo principal: se detiene en vez de seguir con NaN
        raise RuntimeError(f"Error al cargar o evaluar el modelo MoE: {e}") from e
else:
    # Sin datos sintéticos de respaldo: si falta un artefacto de un notebook
    # anterior hay que ejecutarlo, no seguir con datos inventados.
    raise FileNotFoundError(f"No existe {moe_path}. Ejecuta antes el notebook 08 (make pipeline).")


---
## 3. Familia Estadística — 10 Modelos

Se implementan 10 modelos estadísticos/clásicos de series de tiempo. Para aquellos que aceptan features exógenas se utilizan; para los puramente univariados se usa solo la serie `demand`.

### Modelos:
1. **OLS** (Linear Regression con features)
2. **Ridge** (Regularización L2)
3. **Lasso** (Regularización L1)
4. **ElasticNet** (L1+L2)
5. **BayesianRidge**
6. **HuberRegressor** (Robusto a outliers)
7. **AutoReg** (Autoregresivo puro — statsmodels)
8. **Exponential Smoothing** (Holt-Winters aditivo)
9. **SARIMAX** (con exógenas si es factible computacionalmente)
10. **Prophet**

> **Nota**: Los modelos 1-6 son lineales/generalizados con features ingenierizadas; 7-10 son modelos de series de tiempo clásicos.

In [ ]:
# ============================================================
# 4. MODELOS ESTADÍSTICOS — SCREENING
# ============================================================
from sklearn.linear_model import LinearRegression

stat_results = OrderedDict()

# --- 4.1  OLS ---
print("Entrenando: OLS (Linear Regression)...")
with timer_context() as t:
    m = LinearRegression(n_jobs=N_JOBS).fit(X_train, y_train)
stat_results['OLS'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.2  Ridge ---
print("Entrenando: Ridge...")
with timer_context() as t:
    m = Ridge(alpha=1.0, random_state=RANDOM_SEED).fit(X_train, y_train)
stat_results['Ridge'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.3  Lasso ---
print("Entrenando: Lasso...")
with timer_context() as t:
    m = Lasso(alpha=1.0, random_state=RANDOM_SEED, max_iter=5000).fit(X_train, y_train)
stat_results['Lasso'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.4  ElasticNet ---
print("Entrenando: ElasticNet...")
with timer_context() as t:
    m = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=RANDOM_SEED, max_iter=5000).fit(X_train, y_train)
stat_results['ElasticNet'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.5  BayesianRidge ---
print("Entrenando: BayesianRidge...")
with timer_context() as t:
    m = BayesianRidge(max_iter=500).fit(X_train, y_train)
stat_results['BayesianRidge'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.6  HuberRegressor ---
print("Entrenando: HuberRegressor...")
with timer_context() as t:
    m = HuberRegressor(max_iter=500).fit(X_train, y_train)
stat_results['HuberRegressor'] = {
    'model': m,
    'val_metrics': compute_metrics(y_val, m.predict(X_val)),
    'train_time': t.elapsed,
    'type': 'feature_based'
}

# --- 4.7  AutoReg (statsmodels) ---
print("Entrenando: AutoReg (lags=168)...")
try:
    with timer_context() as t:
        ar_model = AutoReg(ts_train.values, lags=168, old_names=False).fit()
    ar_preds_val = ar_model.predict(start=len(ts_train), end=len(ts_train)+len(ts_val)-1)
    stat_results['AutoReg'] = {
        'model': ar_model,
        'val_metrics': compute_metrics(y_val.values, ar_preds_val),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ AutoReg falló: {e}")
    stat_results['AutoReg'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- 4.8  Exponential Smoothing (Holt-Winters) ---
print("Entrenando: Holt-Winters Exponential Smoothing...")
try:
    # Submuestreo diario para viabilidad computacional
    ts_daily_train = ts_train.resample('D').mean()
    ts_daily_val   = ts_val.resample('D').mean()
    with timer_context() as t:
        hw_model = ExponentialSmoothing(
            ts_daily_train, trend='add', seasonal='add',
            seasonal_periods=7
        ).fit(optimized=True)
    hw_preds_daily = hw_model.forecast(len(ts_daily_val))
    # Interpolar a horario para comparación justa
    hw_preds_hourly = np.interp(
        np.linspace(0, len(hw_preds_daily)-1, len(ts_val)),
        np.arange(len(hw_preds_daily)),
        hw_preds_daily.values
    )
    stat_results['HoltWinters'] = {
        'model': hw_model,
        'val_metrics': compute_metrics(y_val.values, hw_preds_hourly),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ HoltWinters falló: {e}")
    stat_results['HoltWinters'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- 4.9  SARIMAX ---
print("Entrenando: SARIMAX (submuestreo diario)...")
try:
    ts_daily_train = ts_train.resample('D').mean()
    ts_daily_val   = ts_val.resample('D').mean()
    with timer_context() as t:
        sarimax_model = SARIMAX(
            ts_daily_train, order=(2,1,2),
            seasonal_order=(1,1,1,7),
            enforce_stationarity=False, enforce_invertibility=False
        ).fit(disp=False, maxiter=200)
    sar_preds_daily = sarimax_model.forecast(len(ts_daily_val))
    sar_preds_hourly = np.interp(
        np.linspace(0, len(sar_preds_daily)-1, len(ts_val)),
        np.arange(len(sar_preds_daily)),
        sar_preds_daily.values
    )
    stat_results['SARIMAX'] = {
        'model': sarimax_model,
        'val_metrics': compute_metrics(y_val.values, sar_preds_hourly),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ SARIMAX falló: {e}")
    stat_results['SARIMAX'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- 4.10  Prophet ---
print("Entrenando: Prophet...")
try:
    from prophet import Prophet
    prophet_train = pd.DataFrame({
        'ds': ts_train.index, 'y': ts_train.values
    })
    with timer_context() as t:
        prophet_model = Prophet(
            daily_seasonality=True, weekly_seasonality=True,
            yearly_seasonality=True, changepoint_prior_scale=0.05
        )
        prophet_model.fit(prophet_train)
    future = prophet_model.make_future_dataframe(periods=len(ts_val), freq='h')
    prophet_preds_all = prophet_model.predict(future)
    prophet_preds_val = prophet_preds_all['yhat'].iloc[-len(ts_val):].values
    stat_results['Prophet'] = {
        'model': prophet_model,
        'val_metrics': compute_metrics(y_val.values, prophet_preds_val),
        'train_time': t.elapsed,
        'type': 'ts_univariate'
    }
except Exception as e:
    print(f"  ⚠️ Prophet falló: {e}")
    stat_results['Prophet'] = {
        'model': None,
        'val_metrics': {'MAE': 1e9, 'RMSE': 1e9, 'MAPE': 100, 'R2': -1},
        'train_time': 0, 'type': 'ts_univariate'
    }

# --- Resultados ---
stat_screening_df = print_screening_results(stat_results, 'Estadística')

---
## 4. Familia Machine Learning — 10 Modelos

Todos los modelos ML operan sobre la misma matriz de features (`X_train`, `X_val`) sin segmentación por tipo de día/semana.

### Modelos:
1. Random Forest
2. Gradient Boosting (sklearn)
3. XGBoost
4. LightGBM
5. Extra Trees
6. AdaBoost
7. SVR (kernel RBF)
8. KNN Regressor
9. Bagging Regressor
10. Decision Tree (baseline)

In [ ]:
# ============================================================
# 4. MODELOS ML — SCREENING
# ============================================================
ml_results = OrderedDict()

ml_models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100, random_state=RANDOM_SEED),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS, verbose=-1),
    'ExtraTrees': ExtraTreesRegressor(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=N_JOBS),
    'AdaBoost': AdaBoostRegressor(
        n_estimators=50, random_state=RANDOM_SEED),
    'SVR': Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf'))
    ]),
    'KNN': KNeighborsRegressor(n_neighbors=10, n_jobs=N_JOBS, weights='distance'),
    'Bagging': BaggingRegressor(
        n_estimators=50, random_state=RANDOM_SEED, n_jobs=N_JOBS),
    'DecisionTree': DecisionTreeRegressor(random_state=RANDOM_SEED),
}

for name, model in ml_models.items():
    print(f"Entrenando: {name}...")
    with timer_context() as t:
        model.fit(X_train, y_train)
    preds_val = model.predict(X_val)
    ml_results[name] = {
        'model': model,
        'val_metrics': compute_metrics(y_val, preds_val),
        'train_time': t.elapsed,
        'type': 'feature_based'
    }

ml_screening_df = print_screening_results(ml_results, 'Machine Learning')

---
## 5. Familia Deep Learning — 10 Modelos (PyTorch)

Se implementan 10 arquitecturas de DL para pronóstico de series de tiempo. Todas comparten:
- Mismo preprocesamiento (StandardScaler)
- Ventana de entrada: 168 pasos (1 semana)
- Horizonte de salida: 1 paso (one-step-ahead, luego rolling)
- Entrenamiento con Adam, MSELoss, early stopping

### Modelos:
1. **LSTM** — Long Short-Term Memory
2. **GRU** — Gated Recurrent Unit
3. **BiLSTM** — Bidirectional LSTM
4. **Stacked LSTM** — 3 capas LSTM
5. **CNN-1D** — Convolutional 1D
6. **TCN** — Temporal Convolutional Network
7. **Transformer** — Encoder-only Transformer
8. **CNN-LSTM** — Híbrido CNN + LSTM
9. **Attention-LSTM** — LSTM con mecanismo de atención
10. **MLP** — Multi-Layer Perceptron (baseline DL)

In [ ]:
# ============================================================
# 5.0  DATASET Y UTILIDADES DL
# ============================================================
WINDOW_SIZE = 168  # 1 semana
DL_EPOCHS = 50
DL_BATCH = 32
DL_LR = 1e-3
DL_PATIENCE = 7
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TimeSeriesDataset(Dataset):
    """Dataset para modelos de secuencia. Ventana deslizante sobre features + target."""
    def __init__(self, X, y, window_size=WINDOW_SIZE):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.X) - self.window_size

    def __getitem__(self, idx):
        x_seq = self.X[idx:idx+self.window_size]
        y_target = self.y[idx+self.window_size]
        return x_seq, y_target

def prepare_dl_data(X_df, y_series, scaler_X, scaler_y, fit=False):
    """Escala y prepara datos para DataLoader."""
    if fit:
        X_scaled = scaler_X.fit_transform(X_df)
        y_scaled = scaler_y.fit_transform(y_series.values.reshape(-1,1)).flatten()
    else:
        X_scaled = scaler_X.transform(X_df)
        y_scaled = scaler_y.transform(y_series.values.reshape(-1,1)).flatten()
    return X_scaled, y_scaled

def train_dl_model(model, train_loader, val_loader, epochs=DL_EPOCHS,
                   lr=DL_LR, patience=DL_PATIENCE):
    """Loop de entrenamiento genérico con early stopping."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.MSELoss()
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(xb).squeeze()
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).squeeze()
                val_losses.append(criterion(pred, yb).item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_state)
    return model, history

def predict_dl(model, data_loader, scaler_y):
    """Genera predicciones y las des-escala."""
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            out = model(xb).squeeze().cpu().numpy()
            preds.append(out)
    preds = np.concatenate(preds)
    return scaler_y.inverse_transform(preds.reshape(-1,1)).flatten()

# Preparar datos
X_train_sc, y_train_sc = prepare_dl_data(X_train, y_train, scaler_X, scaler_y, fit=True)
X_val_sc,   y_val_sc   = prepare_dl_data(X_val,   y_val,   scaler_X, scaler_y)

train_ds = TimeSeriesDataset(X_train_sc, y_train_sc, WINDOW_SIZE)
val_ds   = TimeSeriesDataset(X_val_sc,   y_val_sc,   WINDOW_SIZE)

train_loader = DataLoader(train_ds, batch_size=DL_BATCH, shuffle=False, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=DL_BATCH, shuffle=False)

n_features = X_train.shape[1]
print(f"✓ Datos DL preparados: {len(train_ds)} train / {len(val_ds)} val samples")
print(f"  Features: {n_features} | Window: {WINDOW_SIZE} | Device: {device}")

In [ ]:
# ============================================================
# 5.1  DEFINICIÓN DE ARQUITECTURAS DL
# ============================================================

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

class BiLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class StackedLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden_dim, hidden_dim * 3 // 4, batch_first=True)  # 64 → 48
        self.drop2 = nn.Dropout(dropout)
        self.lstm3 = nn.LSTM(hidden_dim * 3 // 4, hidden_dim // 2, batch_first=True)  # 48 → 32
        self.fc = nn.Linear(hidden_dim // 2, 1)
    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        x, _ = self.lstm2(x); x = self.drop2(x)
        x, _ = self.lstm3(x)
        return self.fc(x[:, -1, :])

class CNN1DModel(nn.Module):
    def __init__(self, input_dim, seq_len=WINDOW_SIZE):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=7, padding=3),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, C, T)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()
    def forward(self, x):
        res = self.residual(x)
        out = self.relu(self.bn1(self.conv1(x)[:, :, :x.size(2)]))
        out = self.drop(out)
        out = self.relu(self.bn2(self.conv2(out)[:, :, :x.size(2)]))
        out = self.drop(out)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_dim, channels=[64, 64, 64], kernel_size=5):  # kernel 3 → 5
        super().__init__()
        layers = []
        in_ch = input_dim
        for i, out_ch in enumerate(channels):
            layers.append(TCNBlock(in_ch, out_ch, kernel_size, dilation=2**i))
            in_ch = out_ch
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.tcn(x)
        x = x[:, :, -1]
        return self.fc(x)

class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=2, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, WINDOW_SIZE, d_model) * 0.1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        x = self.input_proj(x) + self.pos_enc[:, :x.size(1), :]
        x = self.encoder(x)
        return self.fc(x[:, -1, :])

class CNNLSTMModel(nn.Module):
    def __init__(self, input_dim, cnn_out=64, lstm_hidden=64, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, cnn_out, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(cnn_out), nn.Dropout(dropout),
            nn.Conv1d(cnn_out, cnn_out, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(cnn_out, lstm_hidden, batch_first=True,
                            num_layers=2, dropout=dropout)  # 1 → 2 capas + dropout
        self.fc = nn.Linear(lstm_hidden, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x).permute(0, 2, 1)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class AttentionLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2,
                            batch_first=True, dropout=dropout)
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # (B, T, H)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)  # (B, T, 1)
        context = (lstm_out * attn_weights).sum(dim=1)  # (B, H)
        return self.fc(context)

class MLPModel(nn.Module):
    def __init__(self, input_dim, seq_len=WINDOW_SIZE, hidden_dims=[64, 64, 64]):
        super().__init__()
        flat_dim = input_dim * seq_len
        layers = []
        in_dim = flat_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

print("✓ 10 arquitecturas DL definidas")

In [ ]:
# ============================================================
# 5.2  ENTRENAMIENTO — SCREENING DL
# ============================================================
dl_results = OrderedDict()

dl_architectures = {
    'LSTM':          lambda: LSTMModel(n_features),
    'GRU':           lambda: GRUModel(n_features),
    'BiLSTM':        lambda: BiLSTMModel(n_features),
    'StackedLSTM':   lambda: StackedLSTMModel(n_features),
    'CNN1D':         lambda: CNN1DModel(n_features),
    'TCN':           lambda: TCNModel(n_features),
    'Transformer':   lambda: TransformerModel(n_features),
    'CNN_LSTM':      lambda: CNNLSTMModel(n_features),
    'AttentionLSTM': lambda: AttentionLSTMModel(n_features),
    'MLP':           lambda: MLPModel(n_features),
}

for name, model_fn in dl_architectures.items():
    print(f"\nEntrenando: {name}...")
    torch.manual_seed(RANDOM_SEED)
    model = model_fn()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parámetros: {n_params:,}")

    with timer_context() as t:
        model, history = train_dl_model(model, train_loader, val_loader)

    # Predicciones en validación
    y_val_pred = predict_dl(model, val_loader, scaler_y)
    # Ajustar longitud
    y_val_actual = y_val.values[WINDOW_SIZE:WINDOW_SIZE+len(y_val_pred)]

    dl_results[name] = {
        'model': model,
        'val_metrics': compute_metrics(y_val_actual, y_val_pred[:len(y_val_actual)]),
        'train_time': t.elapsed,
        'n_params': n_params,
        'history': history,
        'type': 'dl'
    }
    print(f"  MAPE: {dl_results[name]['val_metrics']['MAPE']:.4f}% | "
          f"RMSE: {dl_results[name]['val_metrics']['RMSE']:.2f} | "
          f"Epochs: {len(history['train_loss'])} | Time: {t.elapsed:.1f}s")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

dl_screening_df = print_screening_results(dl_results, 'Deep Learning')

In [ ]:
torch.cuda.empty_cache()

---
## 7. Selección Top-3 por Familia

Se seleccionan los **3 mejores modelos** de cada familia según MAPE en validación para proceder a la optimización de hiperparámetros.

In [ ]:
# ============================================================
# 7. SELECCIÓN TOP-3
# ============================================================
def select_top_k(results_dict, k=3):
    """Selecciona los k mejores modelos por MAPE en validación."""
    ranked = sorted(results_dict.items(),
                    key=lambda x: x[1]['val_metrics']['MAPE'])
    return OrderedDict(ranked[:k])

top3_stat = select_top_k(stat_results, k=3)
top3_ml   = select_top_k(ml_results,   k=3)
top3_dl   = select_top_k(dl_results,   k=3)

print("=" * 70)
print("TOP-3 SELECCIONADOS POR FAMILIA")
print("=" * 70)
for family, top3 in [('Estadísticos', top3_stat), ('ML', top3_ml), ('DL', top3_dl)]:
    print(f"\n  {family}:")
    for i, (name, r) in enumerate(top3.items(), 1):
        print(f"    {i}. {name:20s} — MAPE: {r['val_metrics']['MAPE']:.4f}%")

In [ ]:
# ============================================================
# 7. SELECCIÓN MANUAL
# ============================================================
def select_models_by_name(results_dict, selected_models):
    """Selecciona modelos específicos por nombre exacto."""
    return OrderedDict(
        (name, results_dict[name])
        for name in selected_models
        if name in results_dict
    )

top3_stat = select_models_by_name(stat_results, ["Lasso"])
top3_ml   = select_models_by_name(ml_results,   ["AdaBoost"])
top3_dl   = select_models_by_name(dl_results,   ["Transformer"])

print("=" * 70)
print("TOP-3 SELECCIONADOS POR FAMILIA")
print("=" * 70)
for family, top3 in [('Estadísticos', top3_stat), ('ML', top3_ml), ('DL', top3_dl)]:
    print(f"\n  {family}:")
    for i, (name, r) in enumerate(top3.items(), 1):
        print(f"    {i}. {name:20s} — MAPE: {r['val_metrics']['MAPE']:.4f}%")

---
## 7. Optimización de Hiperparámetros — Random Search

Según **Bergstra & Bengio (2012)**, el random search es más eficiente que el grid search cuando solo un subconjunto de hiperparámetros es realmente importante (lo cual es el caso típico). Con **$n=60$** evaluaciones aleatorias:

$$P(\text{encontrar top-5\%}) \geq 1 - (1 - 0.05)^{60} \approx 0.954$$

Se utiliza `RandomizedSearchCV` con `TimeSeriesSplit` para respetar la estructura temporal.

In [ ]:
# ============================================================
# 8.1  ESPACIOS DE BÚSQUEDA
# ============================================================
from scipy.stats import uniform, randint, loguniform

# TimeSeriesSplit para validación cruzada temporal
tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS)

# --- Espacios para modelos feature-based (estadísticos lineales + ML) ---
SEARCH_SPACES = {
    # Estadísticos feature-based
    'OLS': None,  # Sin hiperparámetros
    'Ridge': {
        'alpha': loguniform(1e-4, 1e4),
    },
    'Lasso': {
        'alpha': loguniform(1e-4, 1e2),
        'max_iter': [5000, 10000],
    },
    'ElasticNet': {
        'alpha': loguniform(1e-4, 1e2),
        'l1_ratio': uniform(0.01, 0.99),
        'max_iter': [5000, 10000],
    },
    'BayesianRidge': {
        'alpha_1': loguniform(1e-8, 1e-2),
        'alpha_2': loguniform(1e-8, 1e-2),
        'lambda_1': loguniform(1e-8, 1e-2),
        'lambda_2': loguniform(1e-8, 1e-2),
    },
    'HuberRegressor': {
        'epsilon': uniform(1.01, 2.0),
        'alpha': loguniform(1e-5, 1e-1),
        'max_iter': [500, 1000, 2000],
    },
    # ML
    'RandomForest': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(5, 30),
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
        'max_features': ['sqrt', 'log2', 0.5, 0.8, None],
    },
    'GradientBoosting': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 12),
        'learning_rate': loguniform(0.005, 0.3),
        'subsample': uniform(0.6, 0.4),
        'min_samples_split': randint(2, 20),
    },
    'XGBoost': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 12),
        'learning_rate': loguniform(0.005, 0.3),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.5, 0.5),
        'reg_alpha': loguniform(1e-3, 10),
        'reg_lambda': loguniform(1e-3, 10),
    },
    'LightGBM': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(3, 15),
        'learning_rate': loguniform(0.005, 0.3),
        'num_leaves': randint(20, 100),
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.5, 0.5),
        'reg_alpha': loguniform(1e-3, 10),
        'reg_lambda': loguniform(1e-3, 10),
    },
    'ExtraTrees': {
        'n_estimators': randint(100, 500),
        'max_depth': randint(5, 30),
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
    },
    'AdaBoost': {
        'n_estimators': randint(50, 300),
        'learning_rate': loguniform(0.01, 1.0),
    },
    'SVR': {
        'svr__C': loguniform(1, 1000),
        'svr__epsilon': loguniform(0.01, 1),
        'svr__gamma': ['scale', 'auto'],
    },
    'KNN': {
        'n_neighbors': randint(3, 30),
        'weights': ['uniform', 'distance'],
        'p': [1, 2],
    },
    'Bagging': {
        'n_estimators': randint(20, 150),
        'max_samples': uniform(0.5, 0.5),
        'max_features': uniform(0.5, 0.5),
    },
    'DecisionTree': {
        'max_depth': randint(5, 30),
        'min_samples_split': randint(2, 20),
        'min_samples_leaf': randint(1, 10),
    },
}

print("✓ Espacios de búsqueda definidos para", len(SEARCH_SPACES), "modelos")

In [ ]:
# ============================================================
# 7.2  HPO — MODELOS FEATURE-BASED (Estadísticos + ML)
# ============================================================
hpo_results = OrderedDict()

# Combinar top-3 de estadísticos y ML que sean feature-based
top3_feature_based = {}
for name, r in list(top3_stat.items()) + list(top3_ml.items()):
    if r['type'] == 'feature_based' and name in SEARCH_SPACES:
        top3_feature_based[name] = r

print(f"Modelos feature-based para HPO: {list(top3_feature_based.keys())}")
print(f"Usando RandomizedSearchCV con n_iter={N_RANDOM_SEARCH_ITER}, CV={N_CV_FOLDS} folds\n")

for name, r in top3_feature_based.items():
    space = SEARCH_SPACES.get(name)
    if space is None:
        print(f"  {name}: sin hiperparámetros → se mantiene modelo screening")
        hpo_results[name] = r
        continue

    print(f"  Optimizando: {name} ({N_RANDOM_SEARCH_ITER} iteraciones)...")
    base_model = r['model']

    # Para pipelines (SVR), usar el pipeline completo
    if isinstance(base_model, Pipeline):
        estimator = Pipeline([
            ('scaler', StandardScaler()),
            ('svr', SVR(kernel='rbf'))
        ])
    else:
        estimator = base_model.__class__(**{
            k: v for k, v in base_model.get_params().items()
            if k in ['random_state', 'n_jobs', 'verbose', 'verbosity']
            and k in base_model.get_params()
        })
        # Re-set seed params
        try: estimator.set_params(random_state=RANDOM_SEED)
        except: pass
        try: estimator.set_params(n_jobs=N_JOBS)
        except: pass

    with timer_context() as t:
        search = RandomizedSearchCV(
            estimator, space,
            n_iter=N_RANDOM_SEARCH_ITER if name != 'RandomForest' else 10,
            scoring='neg_mean_absolute_percentage_error',
            cv=tscv, random_state=RANDOM_SEED, n_jobs=N_JOBS,
            verbose=0, refit=True
        )
        search.fit(X_trainval, y_trainval)

    # Evaluar en test
    best_model = search.best_estimator_
    preds_test = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test, preds_test)

    hpo_results[name] = {
        'model': best_model,
        'best_params': search.best_params_,
        'val_metrics': r['val_metrics'],  # mantener screening para referencia
        'test_metrics': test_metrics,
        'train_time': t.elapsed,
        'best_cv_score': -search.best_score_,
        'type': r['type']
    }
    print(f"    ✓ Best CV MAPE: {-search.best_score_:.4f}% | Test MAPE: {test_metrics['MAPE']:.4f}%")
    print(f"    Best params: {search.best_params_}")
    print(f"    Tiempo HPO: {t.elapsed:.1f}s\n")

# Modelos estadísticos TS univariados (no aplica RandomizedSearchCV estándar)
for name, r in top3_stat.items():
    if r['type'] == 'ts_univariate' and name not in hpo_results:
        print(f"  {name} (TS univariado): HPO manual no implementado → se mantiene screening")
        hpo_results[name] = r

print(f"\n✓ HPO completado para {len(hpo_results)} modelos feature-based/estadísticos")

In [ ]:
# ============================================================
# 8.3  HPO — MODELOS DEEP LEARNING (Random Search manual)
# ============================================================
# Para DL, se implementa Random Search manual sobre hiperparámetros clave

DL_SEARCH_SPACES = {
    'hidden_dim': [32, 64, 128, 256],
    'num_layers': [1, 2, 3],
    'dropout': [0.1, 0.2, 0.3, 0.4],
    'lr': [5e-4, 1e-3, 2e-3, 5e-3],
    'batch_size': [128, 256, 512],
}

dl_hpo_results = OrderedDict()

# Mapeo de arquitecturas a constructores con parámetros variables
def create_dl_model_with_params(arch_name, input_dim, hidden_dim, num_layers, dropout):
    """Crea modelo DL con hiperparámetros específicos."""
    if arch_name == 'LSTM':
        return LSTMModel(input_dim, hidden_dim, num_layers, dropout)
    elif arch_name == 'GRU':
        return GRUModel(input_dim, hidden_dim, num_layers, dropout)
    elif arch_name == 'BiLSTM':
        return BiLSTMModel(input_dim, hidden_dim // 2, num_layers, dropout)
    elif arch_name == 'StackedLSTM':
        return StackedLSTMModel(input_dim, hidden_dim, dropout)
    elif arch_name == 'CNN1D':
        return CNN1DModel(input_dim)
    elif arch_name == 'TCN':
        return TCNModel(input_dim, channels=[hidden_dim, hidden_dim*2, hidden_dim])
    elif arch_name == 'Transformer':
        # nhead must divide d_model
        d_model = max(32, hidden_dim)
        nhead = 4 if d_model % 4 == 0 else 2
        return TransformerModel(input_dim, d_model=d_model, nhead=nhead,
                                num_layers=num_layers, dropout=dropout)
    elif arch_name == 'CNN_LSTM':
        return CNNLSTMModel(input_dim, cnn_out=hidden_dim, lstm_hidden=hidden_dim)
    elif arch_name == 'AttentionLSTM':
        return AttentionLSTMModel(input_dim, hidden_dim, dropout)
    elif arch_name == 'MLP':
        return MLPModel(input_dim, WINDOW_SIZE, [hidden_dim*2, hidden_dim, hidden_dim//2])
    else:
        raise ValueError(f"Arquitectura desconocida: {arch_name}")

N_DL_RANDOM_TRIALS = 20  # Reducido por costo computacional; escalar si hay GPU

for arch_name in top3_dl.keys():
    print(f"\nHPO para {arch_name} ({N_DL_RANDOM_TRIALS} trials)...")
    best_mape = float('inf')
    best_config = None
    best_model_state = None
    trials_log = []

    for trial in range(N_DL_RANDOM_TRIALS):
        # Samplear hiperparámetros
        config = {
            'hidden_dim': np.random.choice(DL_SEARCH_SPACES['hidden_dim']),
            'num_layers': np.random.choice(DL_SEARCH_SPACES['num_layers']),
            'dropout': np.random.choice(DL_SEARCH_SPACES['dropout']),
            'lr': np.random.choice(DL_SEARCH_SPACES['lr']),
            'batch_size': np.random.choice(DL_SEARCH_SPACES['batch_size']),
        }

        torch.manual_seed(RANDOM_SEED + trial)
        model = create_dl_model_with_params(
            arch_name, n_features,
            int(config['hidden_dim']),
            int(config['num_layers']),
            config['dropout']
        )

        # Reconstruir loaders con nuevo batch_size
        tl = DataLoader(train_ds, batch_size=int(config['batch_size']),
                        shuffle=False, drop_last=True)
        vl = DataLoader(val_ds, batch_size=int(config['batch_size']), shuffle=False)

        try:
            model, hist = train_dl_model(model, tl, vl, epochs=30,
                                         lr=config['lr'], patience=5)
            y_pred = predict_dl(model, vl, scaler_y)
            y_actual = y_val.values[WINDOW_SIZE:WINDOW_SIZE+len(y_pred)]
            mets = compute_metrics(y_actual, y_pred[:len(y_actual)])

            trials_log.append({**config, 'MAPE': mets['MAPE'], 'trial': trial})

            if mets['MAPE'] < best_mape:
                best_mape = mets['MAPE']
                best_config = config
                best_model_state = copy.deepcopy(model.state_dict())
                best_model_arch = model

            if (trial + 1) % 5 == 0:
                print(f"    Trial {trial+1}/{N_DL_RANDOM_TRIALS} | "
                      f"Current MAPE: {mets['MAPE']:.4f}% | Best: {best_mape:.4f}%")
        except Exception as e:
            print(f"    Trial {trial+1} falló: {e}")
            continue

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Reentrenar mejor configuración con más épocas
    print(f"  ✓ Mejor config: {best_config} → MAPE: {best_mape:.4f}%")

    if best_model_state is not None:
        # Reentrenar con train+val (usando todos los datos hasta test)
        X_tv_sc = scaler_X.transform(X_trainval)
        y_tv_sc = scaler_y.transform(y_trainval.values.reshape(-1,1)).flatten()
        tv_ds = TimeSeriesDataset(X_tv_sc, y_tv_sc, WINDOW_SIZE)
        X_test_sc = scaler_X.transform(X_test)
        y_test_sc = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()
        test_ds = TimeSeriesDataset(X_test_sc, y_test_sc, WINDOW_SIZE)

        tv_loader = DataLoader(tv_ds, batch_size=int(best_config['batch_size']),
                               shuffle=False, drop_last=True)
        test_loader_dl = DataLoader(test_ds, batch_size=int(best_config['batch_size']),
                                    shuffle=False)

        torch.manual_seed(RANDOM_SEED)
        final_model = create_dl_model_with_params(
            arch_name, n_features,
            int(best_config['hidden_dim']),
            int(best_config['num_layers']),
            best_config['dropout']
        )
        with timer_context() as t_retrain:
            final_model, _ = train_dl_model(
                final_model, tv_loader, test_loader_dl,
                epochs=DL_EPOCHS, lr=best_config['lr'], patience=DL_PATIENCE
            )

        # Predicciones en test
        test_preds = predict_dl(final_model, test_loader_dl, scaler_y)
        y_test_actual = y_test.values[WINDOW_SIZE:WINDOW_SIZE+len(test_preds)]
        test_mets = compute_metrics(y_test_actual, test_preds[:len(y_test_actual)])

        dl_hpo_results[arch_name] = {
            'model': final_model,
            'best_params': best_config,
            'test_metrics': test_mets,
            'val_metrics': top3_dl[arch_name]['val_metrics'],
            'train_time': t_retrain.elapsed,
            'trials_log': trials_log,
            'type': 'dl'
        }
        print(f"  ✓ Test MAPE: {test_mets['MAPE']:.4f}% | "
              f"RMSE: {test_mets['RMSE']:.2f} | Time: {t_retrain.elapsed:.1f}s")

print(f"\n✓ HPO DL completado para {len(dl_hpo_results)} modelos")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import pickle
import pandas as pd
from collections import OrderedDict

# List of variables to export and their types
variables_to_export = {
    'stat_screening_df': {'obj': stat_screening_df, 'type': 'dataframe'},
    'ml_screening_df': {'obj': ml_screening_df, 'type': 'dataframe'},
    'dl_screening_df': {'obj': dl_screening_df, 'type': 'dataframe'},
    'stat_results': {'obj': stat_results, 'type': 'pickle'},
    'ml_results': {'obj': ml_results, 'type': 'pickle'},
    'dl_results': {'obj': dl_results, 'type': 'pickle'},
    'top3_stat': {'obj': top3_stat, 'type': 'pickle'},
    'top3_ml': {'obj': top3_ml, 'type': 'pickle'},
    'top3_dl': {'obj': top3_dl, 'type': 'pickle'},
    'hpo_results': {'obj': hpo_results, 'type': 'pickle'},
    'dl_hpo_results': {'obj': dl_hpo_results, 'type': 'pickle'},
}

print(f"Exportando variables a: {BENCH_DIR}")

for var_name, var_info in variables_to_export.items():
    obj = var_info['obj']
    obj_type = var_info['type']

    try:
        if obj_type == 'dataframe':
            filepath = BENCH_DIR / f'{var_name}.csv'
            obj.to_csv(filepath, index=False)
            print(f"  ✓ Exportado {var_name} como CSV a {filepath}")
        elif obj_type == 'pickle':
            filepath = BENCH_DIR / f'{var_name}.pkl'
            with open(filepath, 'wb') as f:
                pickle.dump(obj, f)
            print(f"  ✓ Exportado {var_name} como PKL a {filepath}")
        else:
            print(f"  ✗ Tipo de objeto no soportado para {var_name}: {type(obj)}")
    except Exception as e:
        print(f"  ❌ Error al exportar {var_name}: {e}")

print("\n✓ Exportación de variables completada.")

---
## 8. Evaluación Final en Test Set

Se evalúan todos los modelos optimizados + MoE en el test set con las mismas métricas.

In [ ]:
# ============================================================
# 9. EVALUACIÓN FINAL — TODOS LOS MODELOS EN TEST
# ============================================================
final_results = OrderedDict()

# --- MoE ---
final_results['MoE (Prop)'] = {
    'family': 'MoE',
    'test_metrics': moe_metrics_test,
    'train_time': 20.7954, #moe_train_time
    'infer_time': 0.2859, #moe_infer_time
}

# --- Modelos feature-based optimizados ---
for name, r in hpo_results.items():
    if r['type'] == 'feature_based':
        model = r['model']
        with timer_context() as t_infer:
            preds = model.predict(X_test)
        test_mets = compute_metrics(y_test, preds) if 'test_metrics' not in r else r['test_metrics']
        final_results[f'{name} (Opt)'] = {
            'family': 'Estadístico' if name in stat_results else 'Machine Learning',
            'test_metrics': test_mets,
            'train_time': r['train_time'],
            'infer_time': t_infer.elapsed,
            'predictions': preds,
        }
    elif r['type'] == 'ts_univariate':
        # Para modelos TS, usar métricas de validación como proxy
        final_results[f'{name} (TS)'] = {
            'family': 'Estadístico',
            'test_metrics': r.get('test_metrics', r['val_metrics']),
            'train_time': r['train_time'],
            'infer_time': 0, # Para modelos TS clásicos, la inferencia es instantánea o trivial
        }

# --- Modelos DL optimizados ---
# Preparar test_loader para DL para medir inferencia
X_test_sc_dl = scaler_X.transform(X_test)
y_test_sc_dl = scaler_y.transform(y_test.values.reshape(-1,1)).flatten() # Dummy y_test_sc_dl, no se usa en predict_dl
test_ds_dl = TimeSeriesDataset(X_test_sc_dl, y_test_sc_dl, WINDOW_SIZE)
test_loader_dl = DataLoader(test_ds_dl, batch_size=DL_BATCH, shuffle=False)

for name, r in dl_hpo_results.items():
    model = r['model']
    with timer_context() as t_infer_dl:
        # Necesitamos el predict_dl para obtener las predicciones y medir el tiempo
        preds_dl = predict_dl(model, test_loader_dl, scaler_y)
    # Ajustar longitud de y_test_actual para el cálculo de métricas
    y_test_actual_dl = y_test.values[WINDOW_SIZE : WINDOW_SIZE + len(preds_dl)]

    final_results[f'{name} (Opt)'] = {
        'family': 'Deep Learning',
        'test_metrics': compute_metrics(y_test_actual_dl, preds_dl[:len(y_test_actual_dl)]),
        'train_time': r['train_time'],
        'infer_time': t_infer_dl.elapsed,
    }

# --- Tabla resumen ---
summary_rows = []
for name, r in final_results.items():
    m = r['test_metrics']
    summary_rows.append({
        'Modelo': name,
        'Familia': r['family'],
        'MAPE (%)': m['MAPE'],
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
        'R²': m['R2'],
        'Tiempo Train (s)': r['train_time'],
        'Tiempo Infer (s)': r.get('infer_time', np.nan),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('MAPE (%)')
print("=" * 100)
print("  EVALUACIÓN FINAL EN TEST SET — TODOS LOS MODELOS")
print("=" * 100)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))

# Guardar
summary_df.to_csv(BENCH_DIR / 'benchmark_summary.csv', index=False)
print(f"\n✓ Resultados guardados en {BENCH_DIR / 'benchmark_summary.csv'}")

---
## 9. Análisis Comparativo

### 9.1 Comparación de Desempeño Predictivo

In [ ]:
# ============================================================
# 9.1  VISUALIZACIÓN DE DESEMPEÑO PREDICTIVO
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Comparación de Desempeño Predictivo — Test Set', fontsize=16, fontweight='bold')

# Paleta por familia
family_colors = {'MoE': '#E63946', 'Estadístico': '#457B9D',
                 'Machine Learning': '#2A9D8F', 'Deep Learning': '#E9C46A'}
colors = [family_colors.get(r['family'], '#999') for _, r in final_results.items()]
names = list(final_results.keys())

metrics_to_plot = ['MAPE (%)', 'RMSE', 'MAE', 'R²']
for ax, metric in zip(axes.flatten(), metrics_to_plot):
    values = summary_df[metric].values
    bars = ax.barh(summary_df['Modelo'], values, color=[
        family_colors.get(f, '#999') for f in summary_df['Familia']
    ], edgecolor='white', linewidth=0.5)
    ax.set_xlabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.invert_yaxis()
    # Anotar valores
    for bar, val in zip(bars, values):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                f'  {val:.4f}', va='center', fontsize=8)

# Leyenda
patches = [mpatches.Patch(color=c, label=f) for f, c in family_colors.items()]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(BENCH_DIR / 'fig_predictive_performance.png', bbox_inches='tight')
plt.show()
print("✓ Figura guardada: fig_predictive_performance.png")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Predictive Performance Comparison — Test Set', fontsize=16, fontweight='bold')

family_colors = {'MoE': '#E63946', 'Estadístico': '#457B9D',
                 'Machine Learning': '#2A9D8F', 'Deep Learning': '#E9C46A'}

metrics_to_plot = ['MAPE (%)', 'RMSE', 'MAE', 'R²']
for ax, metric in zip(axes.flatten(), metrics_to_plot):
    values = summary_df[metric].values
    bars = ax.barh(summary_df['Modelo'], values, color=[
        family_colors.get(f, '#999') for f in summary_df['Familia']
    ], edgecolor='white', linewidth=0.5)
    ax.set_xlabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.invert_yaxis()
    for bar, val in zip(bars, values):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
                f'  {val:.4f}', va='center', fontsize=8)

patches = [mpatches.Patch(color=c, label=f) for f, c in {
    'MoE': '#E63946',
    'Statistical': '#457B9D',
    'Machine Learning': '#2A9D8F',
    'Deep Learning': '#E9C46A'
}.items()]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(BENCH_DIR / 'english/fig_predictive_performance.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved: fig_predictive_performance.png")

In [ ]:
# ============================================================
# 9.1b  RADAR CHART — Perfil de cada modelo
# ============================================================
from matplotlib.patches import FancyBboxPatch

# Normalizar métricas para radar (0-1, donde 1 es mejor)
radar_df = summary_df.copy()
for col in ['MAPE (%)', 'RMSE', 'MAE']:
    vmin, vmax = radar_df[col].min(), radar_df[col].max()
    if vmax > vmin:
        radar_df[col + '_norm'] = 1 - (radar_df[col] - vmin) / (vmax - vmin)  # invertir: menor es mejor
    else:
        radar_df[col + '_norm'] = 1.0
radar_df['R²_norm'] = (radar_df['R²'] - radar_df['R²'].min()) / (
    radar_df['R²'].max() - radar_df['R²'].min()) if radar_df['R²'].max() > radar_df['R²'].min() else 1.0

categories = ['MAPE', 'RMSE', 'MAE', 'R²']
norm_cols = ['MAPE (%)_norm', 'RMSE_norm', 'MAE_norm', 'R²_norm']

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

for i, row in radar_df.head(6).iterrows():  # Top 6 modelos
    values = [row[c] for c in norm_cols]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Modelo'])
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_title('Perfil Multimétrica — Top 6 Modelos', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_radar_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.patches import FancyBboxPatch

radar_df = summary_df.copy()
for col in ['MAPE (%)', 'RMSE', 'MAE']:
    vmin, vmax = radar_df[col].min(), radar_df[col].max()
    if vmax > vmin:
        radar_df[col + '_norm'] = 1 - (radar_df[col] - vmin) / (vmax - vmin)
    else:
        radar_df[col + '_norm'] = 1.0
radar_df['R²_norm'] = (radar_df['R²'] - radar_df['R²'].min()) / (
    radar_df['R²'].max() - radar_df['R²'].min()) if radar_df['R²'].max() > radar_df['R²'].min() else 1.0

categories = ['MAPE', 'RMSE', 'MAE', 'R²']
norm_cols = ['MAPE (%)_norm', 'RMSE_norm', 'MAE_norm', 'R²_norm']

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

for i, row in radar_df.head(6).iterrows():
    values = [row[c] for c in norm_cols]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Modelo'])
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_title('Multi-Metric Profile — Top 6 Models', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_radar_comparison.png', bbox_inches='tight')
plt.show()

### 9.2 Comparación de Eficiencia Computacional

In [ ]:
# ============================================================
# 9.2  EFICIENCIA COMPUTACIONAL
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Tiempo de entrenamiento
ax = axes[0]
train_times = summary_df.sort_values('Tiempo Train (s)')
colors_t = [family_colors.get(f, '#999') for f in train_times['Familia']]
ax.barh(train_times['Modelo'], train_times['Tiempo Train (s)'],
        color=colors_t, edgecolor='white')
ax.set_xlabel('Tiempo de Entrenamiento (s)')
ax.set_title('Tiempo de Entrenamiento', fontweight='bold')
ax.invert_yaxis()
for i, (_, row) in enumerate(train_times.iterrows()):
    ax.text(row['Tiempo Train (s)'], i, f"  {row['Tiempo Train (s)']:.1f}s",
            va='center', fontsize=8)

# Trade-off: MAPE vs Tiempo
ax = axes[1]
for family, color in family_colors.items():
    mask = summary_df['Familia'] == family
    sub = summary_df[mask]
    ax.scatter(sub['Tiempo Train (s)'], sub['MAPE (%)'],
               c=color, s=120, label=family, edgecolors='black', zorder=5)
    for _, row in sub.iterrows():
        ax.annotate(row['Modelo'].split(' (')[0], (row['Tiempo Train (s)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
ax.set_xlabel('Tiempo de Entrenamiento (s)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Trade-off: Precisión vs Eficiencia', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_computational_efficiency_train.png', bbox_inches='tight')
plt.show()
print("✓ Figura guardada: fig_computational_efficiency_train.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
train_times = summary_df.sort_values('Tiempo Train (s)')
colors_t = [family_colors.get(f, '#999') for f in train_times['Familia']]
ax.barh(train_times['Modelo'], train_times['Tiempo Train (s)'],
        color=colors_t, edgecolor='white')
ax.set_xlabel('Training Time (s)')
ax.set_title('Training Time', fontweight='bold')
ax.invert_yaxis()
for i, (_, row) in enumerate(train_times.iterrows()):
    ax.text(row['Tiempo Train (s)'], i, f"  {row['Tiempo Train (s)']:.1f}s",
            va='center', fontsize=8)

ax = axes[1]
for family, color in family_colors.items():
    mask = summary_df['Familia'] == family
    sub = summary_df[mask]
    ax.scatter(sub['Tiempo Train (s)'], sub['MAPE (%)'],
               c=color, s=120, label=family, edgecolors='black', zorder=5)
    for _, row in sub.iterrows():
        ax.annotate(row['Modelo'].split(' (')[0], (row['Tiempo Train (s)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
ax.set_xlabel('Inference Time (s)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Trade-off: Accuracy vs Efficiency', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_computational_efficiency_train.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved: fig_computational_efficiency_train.png")

In [ ]:
# ============================================================
# 9.2  EFICIENCIA COMPUTACIONAL
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Tiempo de Inferencia
ax = axes[0]
train_times = summary_df.sort_values('Tiempo Infer (s)')
colors_t = [family_colors.get(f, '#999') for f in train_times['Familia']]
ax.barh(train_times['Modelo'], train_times['Tiempo Infer (s)'],
        color=colors_t, edgecolor='white')
ax.set_xlabel('Tiempo de Inferencia (s)')
ax.set_title('Tiempo de Inferencia', fontweight='bold')
ax.invert_yaxis()
for i, (_, row) in enumerate(train_times.iterrows()):
    ax.text(row['Tiempo Infer (s)'], i, f"  {row['Tiempo Infer (s)']:.1f}s",
            va='center', fontsize=8)

# Trade-off: MAPE vs Tiempo
ax = axes[1]
for family, color in family_colors.items():
    mask = summary_df['Familia'] == family
    sub = summary_df[mask]
    ax.scatter(sub['Tiempo Infer (s)'], sub['MAPE (%)'],
               c=color, s=120, label=family, edgecolors='black', zorder=5)
    for _, row in sub.iterrows():
        ax.annotate(row['Modelo'].split(' (')[0], (row['Tiempo Infer (s)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
ax.set_xlabel('Tiempo de Inferencia (s)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Trade-off: Precisión vs Eficiencia', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_computational_efficiency_inf.png', bbox_inches='tight')
plt.show()
print("✓ Figura guardada: fig_computational_efficiency_inf.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
train_times = summary_df.sort_values('Tiempo Infer (s)')
colors_t = [family_colors.get(f, '#999') for f in train_times['Familia']]
ax.barh(train_times['Modelo'], train_times['Tiempo Infer (s)'],
        color=colors_t, edgecolor='white')
ax.set_xlabel('Inference Time (s)')
ax.set_title('Inference Time', fontweight='bold')
ax.invert_yaxis()
for i, (_, row) in enumerate(train_times.iterrows()):
    ax.text(row['Tiempo Infer (s)'], i, f"  {row['Tiempo Infer (s)']:.1f}s",
            va='center', fontsize=8)

ax = axes[1]
for family, color in family_colors.items():
    mask = summary_df['Familia'] == family
    sub = summary_df[mask]
    ax.scatter(sub['Tiempo Infer (s)'], sub['MAPE (%)'],
               c=color, s=120, label=family, edgecolors='black', zorder=5)
    for _, row in sub.iterrows():
        ax.annotate(row['Modelo'].split(' (')[0], (row['Tiempo Infer (s)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom', xytext=(5, 5),
                    textcoords='offset points')
ax.set_xlabel('Inference Time (s)')
ax.set_ylabel('MAPE (%)')
ax.set_title('Trade-off: Accuracy vs Efficiency', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_computational_efficiency_inf.png', bbox_inches='tight')
plt.show()
print("✓ Figure saved: fig_computational_efficiency_inf.png")

In [ ]:
# ============================================================
# 9.2b  TABLA DE ESCALABILIDAD
# ============================================================
scalability_rows = []
for name, r in final_results.items():
    n_train_samples = len(X_trainval)
    train_time = r['train_time'] if r['train_time'] else 0
    throughput = n_train_samples / train_time if train_time > 0 else np.inf
    scalability_rows.append({
        'Modelo': name,
        'Familia': r['family'],
        'Muestras Train': n_train_samples,
        'Tiempo Train (s)': train_time,
        'Throughput (samples/s)': throughput,
        'MAPE (%)': r['test_metrics']['MAPE'],
    })

scalability_df = pd.DataFrame(scalability_rows).sort_values('Throughput (samples/s)', ascending=False)
print("\n" + "="*80)
print("  ESCALABILIDAD — Throughput de entrenamiento")
print("="*80)
print(scalability_df.to_string(index=False, float_format='{:.2f}'.format))
scalability_df.to_csv(BENCH_DIR / 'scalability_analysis.csv', index=False)

### 9.3 Comparación de Interpretabilidad

Se implementa un análisis cuantitativo de interpretabilidad en **cinco dimensiones**, alineado con la escala definida en el Estado del Arte:

| Puntaje | Significado |
|---------|-------------|
| **1** | Caja negra sin explicación operativa útil |
| **2** | Explicabilidad limitada, dependiente de técnicas auxiliares (SHAP, LIME) sin garantía de trazabilidad suficiente |
| **3** | Interpretabilidad parcial: métricas internas (feature importance, pesos de atención) pero sin trazabilidad completa |
| **4** | Alta interpretabilidad: componentes del modelo directamente analizables (expertos especializados por dominios) |
| **5** | Interpretabilidad total: coeficientes, reglas o componentes directamente legibles y auditables |

#### Dimensiones del análisis:
1. **Escala cuantitativa 1-5** — Puntaje fundamentado por familia
2. **SHAP values** — Contribución de cada feature al pronóstico (todos los modelos)
3. **Feature importance nativa** — Coeficientes e importancias internas
4. **t-SNE/UMAP de asignación de expertos** — Visualización del routing del MoE
5. **Estabilidad del routing** — Consistencia de asignación para muestras similares

#### 9.3a Escala Cuantitativa de Interpretabilidad (1-5)

In [ ]:
# ============================================================
# 9.3a  ESCALA CUANTITATIVA DE INTERPRETABILIDAD (1-5)
# ============================================================
# Escala según articulo Estado del Arte
# Pronóstico de Demanda de Energía Eléctrica
# ============================================================

def assign_interpretability_score(model_name, family):
    """
    Asigna puntaje 1-5 según la escala del Estado del Arte.

    Criterios:
    - 5: Coeficientes/reglas directamente legibles (Regresión Lineal, Lasso, Ridge)
    - 4: Componentes analizables por dominio (MoE con expertos + gating explícito)
    - 3: Métricas internas parciales (tree importance, attention weights)
    - 2: Requiere técnicas post-hoc (SHAP/LIME) para explicar
    - 1: Caja negra sin explicación útil
    """
    scores = {
        # Estadísticos lineales → coeficientes directamente auditables
        'OLS': 5, 'Ridge': 5, 'Lasso': 5, 'ElasticNet': 5,
        'BayesianRidge': 5, 'HuberRegressor': 5,
        # TS clásicos → componentes (tendencia, estacionalidad) legibles
        'AutoReg': 5, 'HoltWinters': 5, 'SARIMAX': 4, 'Prophet': 4,
        # ML tree-based → feature importance nativa, pero no coeficientes directos
        'RandomForest': 3, 'GradientBoosting': 3, 'XGBoost': 3,
        'LightGBM': 3, 'ExtraTrees': 3, 'AdaBoost': 3,
        'DecisionTree': 4,  # Árbol único → reglas directamente legibles
        # ML otros → requieren SHAP/LIME
        'SVR': 2, 'KNN': 2, 'Bagging': 2,
        # DL → caja negra, interpretabilidad limitada
        'LSTM': 2, 'GRU': 2, 'BiLSTM': 2, 'StackedLSTM': 1,
        'CNN1D': 1, 'TCN': 1, 'Transformer': 2,  # attention weights parciales
        'CNN_LSTM': 1, 'AttentionLSTM': 2, 'MLP': 1,
        # MoE → expertos especializados por dominio + gating explícito
        'MoE': 4,
    }
    # Buscar por nombre base (sin sufijos como "(Opt)", "(DL-Opt)")
    for key in scores:
        if key in model_name:
            return scores[key]
    return 2  # default conservador

# Construir tabla
interp_rows = []
for name, r in final_results.items():
    family = r['family']
    score = assign_interpretability_score(name, family)

    # Justificación según score
    justifications = {
        5: 'Coeficientes/componentes directamente legibles y auditables',
        4: 'Componentes del modelo directamente analizables por dominio',
        3: 'Feature importance nativa disponible, pero sin trazabilidad completa',
        2: 'Requiere técnicas post-hoc (SHAP, LIME) para explicabilidad',
        1: 'Caja negra: pesos opacos, sin explicación operativa directa',
    }

    interp_rows.append({
        'Modelo': name,
        'Familia': family,
        'Score (1-5)': score,
        'Justificación': justifications[score],
        'MAPE (%)': r['test_metrics']['MAPE'],
    })

interp_score_df = pd.DataFrame(interp_rows).sort_values('Score (1-5)', ascending=False)

print("=" * 110)
print("  ESCALA CUANTITATIVA DE INTERPRETABILIDAD (1-5)")
print("  Escala: Torres-Sánchez et al. — Estado del Arte Pronóstico Demanda Eléctrica")
print("=" * 110)
print(interp_score_df.to_string(index=False))

# Visualización: Interpretabilidad vs MAPE (trade-off)
fig, ax = plt.subplots(figsize=(12, 7))
for family, color in family_colors.items():
    mask = interp_score_df['Familia'] == family
    sub = interp_score_df[mask]
    ax.scatter(sub['Score (1-5)'], sub['MAPE (%)'],
               c=color, s=150, label=family, edgecolors='black', zorder=5, alpha=0.85)
    for _, row in sub.iterrows():
        ax.annotate(row['Modelo'].split(' (')[0],
                    (row['Score (1-5)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(6, 4), textcoords='offset points')

ax.set_xlabel('Interpretabilidad (1=Caja Negra → 5=Totalmente Auditable)', fontsize=12)
ax.set_ylabel('MAPE (%)', fontsize=12)
ax.set_title('Trade-off: Interpretabilidad vs. Desempeño Predictivo', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1\nCaja negra', '2\nPost-hoc\n(SHAP/LIME)',
                     '3\nFeature imp.\nparcial', '4\nComponentes\nanalizables',
                     '5\nTotalmente\nauditable'], fontsize=9)
ax.legend(loc='upper right')
ax.invert_yaxis()  # Menor MAPE = mejor → arriba
plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_interpretability_vs_performance.png', bbox_inches='tight')
plt.show()

interp_score_df.to_csv(BENCH_DIR / 'interpretability_scores.csv', index=False)
print("\n✓ Tabla y figura guardadas")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
for family, color in family_colors.items():
    mask = interp_score_df['Familia'] == family
    sub = interp_score_df[mask]
    ax.scatter(sub['Score (1-5)'], sub['MAPE (%)'],
               c=color, s=150, label=family, edgecolors='black', zorder=5, alpha=0.85)
    for _, row in sub.iterrows():
        ax.annotate(row['Modelo'].split(' (')[0],
                    (row['Score (1-5)'], row['MAPE (%)']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(6, 4), textcoords='offset points')

ax.set_xlabel('Interpretability (1 = Black Box → 5 = Fully Auditable)', fontsize=12)
ax.set_ylabel('MAPE (%)', fontsize=12)
ax.set_title('Trade-off: Interpretability vs. Predictive Performance', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1\nBlack box', '2\nPost-hoc\n(SHAP/LIME)',
                     '3\nPartial feature\nimportance', '4\nAnalyzable\ncomponents',
                     '5\nFully\nauditable'], fontsize=9)
ax.legend(loc='upper right')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_interpretability_vs_performance.png', bbox_inches='tight')
plt.show()

interp_score_df.to_csv(BENCH_DIR / 'interpretability_scores.csv', index=False)
print("\n✓ Table and figure saved")

#### 9.3b SHAP Values — Contribución de Features por Modelo

Se utiliza SHAP (SHapley Additive exPlanations) para cuantificar la contribución marginal de cada feature al pronóstico. Esto permite comparar **qué información usa cada modelo** de forma agnóstica a la arquitectura.

Para modelos DL se usa `DeepExplainer` o `KernelExplainer` según disponibilidad.

In [ ]:
import shap
shap.initjs()

In [ ]:
# ============================================================
# 9.3b  SHAP VALUES — TODOS LOS MODELOS
# ============================================================

# Submuestreo para eficiencia computacional
N_SHAP_SAMPLES = 500  # Background
N_SHAP_EXPLAIN = 200  # Muestras a explicar
np.random.seed(RANDOM_SEED)

shap_background = X_test.sample(n=min(N_SHAP_SAMPLES, len(X_test)), random_state=RANDOM_SEED)
shap_explain    = X_test.sample(n=min(N_SHAP_EXPLAIN, len(X_test)), random_state=RANDOM_SEED + 1)

shap_values_dict = {}

# --- SHAP para modelos feature-based (TreeExplainer o KernelExplainer) ---
for name, r in hpo_results.items():
    if r['type'] != 'feature_based':
        continue
    model = r['model']
    print(f"SHAP para {name}...", end=" ")

    try:
        # TreeExplainer para tree-based (mucho más rápido)
        if hasattr(model, 'feature_importances_') or name in ['RandomForest', 'GradientBoosting',
            'XGBoost', 'LightGBM', 'ExtraTrees', 'DecisionTree']:
            try:
                explainer = shap.TreeExplainer(model)
                sv = explainer.shap_values(shap_explain)
            except:
                explainer = shap.KernelExplainer(model.predict, shap_background.values[:100])
                sv = explainer.shap_values(shap_explain.values[:50])
        elif isinstance(model, Pipeline):
            # Para pipelines con scaler (SVR, etc.) → KernelExplainer
            explainer = shap.KernelExplainer(model.predict, shap_background.values[:100])
            sv = explainer.shap_values(shap_explain.values[:50])
        else:
            # Modelos lineales → LinearExplainer
            try:
                masker = shap.maskers.Independent(shap_background, max_samples=100)
                explainer = shap.LinearExplainer(model, masker)
                sv = explainer.shap_values(shap_explain)
            except:
                explainer = shap.KernelExplainer(model.predict, shap_background.values[:100])
                sv = explainer.shap_values(shap_explain.values[:50])

        shap_values_dict[name] = sv
        print("✓")
    except Exception as e:
        print(f"⚠️ {e}")

print(f"\n✓ SHAP calculado para {len(shap_values_dict)} modelos feature-based")

In [ ]:
# ============================================================
# 9.3b.2  SHAP PARA MODELOS DL (KernelExplainer)
# ============================================================
# Para DL se usa KernelExplainer con un wrapper que aplana la ventana temporal

N_DL_SHAP_BG = 50      # Background reducido por costo
N_DL_SHAP_EXPLAIN = 30  # Muestras a explicar

for arch_name, r in dl_hpo_results.items():
    print(f"SHAP para {arch_name} (DL)...", end=" ")
    model = r['model']
    model.eval()

    try:
        # Wrapper: recibe features planas de 1 timestep, construye ventana con padding
        def dl_predict_wrapper(X_flat):
            """Predice usando el último timestep de una ventana artificial."""
            X_sc = scaler_X.transform(X_flat)
            # Crear ventana: repetir el mismo vector WINDOW_SIZE veces (aproximación)
            X_seq = np.tile(X_sc[:, np.newaxis, :], (1, WINDOW_SIZE, 1))
            X_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)
            with torch.no_grad():
                preds_sc = model(X_tensor).squeeze().cpu().numpy()
            return scaler_y.inverse_transform(preds_sc.reshape(-1, 1)).flatten()

        bg_sample = shap_background.values[:N_DL_SHAP_BG]
        explain_sample = shap_explain.values[:N_DL_SHAP_EXPLAIN]

        explainer = shap.KernelExplainer(dl_predict_wrapper, bg_sample)
        sv = explainer.shap_values(explain_sample, nsamples=100)
        shap_values_dict[arch_name] = sv
        print("✓")
    except Exception as e:
        print(f"⚠️ {e}")

print(f"\n✓ SHAP total: {len(shap_values_dict)} modelos")

In [ ]:
# ============================================================
# 9.3b.3  VISUALIZACIÓN SHAP — Summary plots comparativos
# ============================================================
# Seleccionar modelos representativos (1 por familia + MoE)
representative_models = []
for family_target in ['Estadístico', 'Machine Learning', 'Deep Learning']:
    for name in shap_values_dict:
        full_name = f'{name} (Opt)' if name in hpo_results else f'{name} (Opt)'
        if full_name in final_results and final_results[full_name]['family'] == family_target:
            representative_models.append(name)
            break
        elif name in final_results and final_results[name]['family'] == family_target:
            representative_models.append(name)
            break

n_rep = len(representative_models)
if n_rep > 0:
    # Ajustar número de columnas para plots, máximo 3 por fila para visibilidad
    ncols = min(n_rep, 3)
    nrows = (n_rep + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 8 * nrows))

    # Asegurar que 'axes' sea un array 1D si nrows es 1 para facilitar la iteración
    if nrows == 1 and ncols > 1: # Si es una sola fila con múltiples columnas
        axes = axes
    elif nrows == 1 and ncols == 1: # Si es un solo plot
        axes = [axes]
    else: # Si son múltiples filas
        axes = axes.flatten()

    for ax_idx, name in enumerate(representative_models):
        ax = axes[ax_idx]
        sv = shap_values_dict[name]
        # Asegurar dimensiones correctas
        if isinstance(sv, list):
            sv = sv[0] if len(sv) > 0 else np.zeros((1, len(feature_cols))) # Fallback para SV vacío
        n_samples_sv = sv.shape[0]
        explain_data = shap_explain.iloc[:n_samples_sv] if n_samples_sv <= len(shap_explain) else shap_explain

        plt.sca(ax)
        shap.summary_plot(sv, explain_data, plot_type='bar',
                          max_display=15, show=False, plot_size=None)
        ax.set_title(f'{name}', fontweight='bold', fontsize=12)
        ax.set_xlabel('mean |SHAP value|')
        ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0)) # Forzar notación científica

    # Eliminar ejes vacíos si hay más subplots que modelos
    for i in range(ax_idx + 1, len(axes)):
        fig.delaxes(axes[i])

    fig.suptitle('SHAP Feature Importance — Representantes por Familia',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_shap_summary_comparison.png', bbox_inches='tight')
    plt.show()

# --- SHAP beeswarm para el mejor modelo global ---
best_model_name = summary_df.iloc[1]['Modelo'].split(' (')[0]
if best_model_name in shap_values_dict:
    sv_best = shap_values_dict[best_model_name]
    if isinstance(sv_best, list):
        sv_best = sv_best[0]
    n_sv = sv_best.shape[0]
    fig, ax = plt.subplots(figsize=(10, 8))
    plt.sca(ax)
    shap.summary_plot(sv_best, shap_explain.iloc[:n_sv],
                      max_display=20, show=False, plot_size=None)
    ax.set_title(f'SHAP Beeswarm — {best_model_name} (Mejor Modelo Global)',
                 fontweight='bold', fontsize=13)
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0)) # Forzar notación científica
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_shap_beeswarm_best.png', bbox_inches='tight')
    plt.show()

print("✓ Figuras SHAP guardadas")

In [ ]:
n_rep = len(representative_models)
if n_rep > 0:
    ncols = min(n_rep, 3)
    nrows = (n_rep + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 8 * nrows))

    if nrows == 1 and ncols > 1:
        axes = axes
    elif nrows == 1 and ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for ax_idx, name in enumerate(representative_models):
        ax = axes[ax_idx]
        sv = shap_values_dict[name]
        if isinstance(sv, list):
            sv = sv[0] if len(sv) > 0 else np.zeros((1, len(feature_cols)))
        n_samples_sv = sv.shape[0]
        explain_data = shap_explain.iloc[:n_samples_sv] if n_samples_sv <= len(shap_explain) else shap_explain

        plt.sca(ax)
        shap.summary_plot(sv, explain_data, plot_type='bar',
                          max_display=15, show=False, plot_size=None)
        ax.set_title(f'{name}', fontweight='bold', fontsize=12)
        ax.set_xlabel('mean |SHAP value|')
        ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))

    for i in range(ax_idx + 1, len(axes)):
        fig.delaxes(axes[i])

    fig.suptitle('SHAP Feature Importance — Family Representatives',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'english/fig_shap_summary_comparison.png', bbox_inches='tight')
    plt.show()

best_model_name = summary_df.iloc[1]['Modelo'].split(' (')[0]
if best_model_name in shap_values_dict:
    sv_best = shap_values_dict[best_model_name]
    if isinstance(sv_best, list):
        sv_best = sv_best[0]
    n_sv = sv_best.shape[0]
    fig, ax = plt.subplots(figsize=(10, 8))
    plt.sca(ax)
    shap.summary_plot(sv_best, shap_explain.iloc[:n_sv],
                      max_display=20, show=False, plot_size=None)
    ax.set_title(f'SHAP Beeswarm — {best_model_name} (Best Overall Model)',
                 fontweight='bold', fontsize=13)
    ax.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'english/fig_shap_beeswarm_best.png', bbox_inches='tight')
    plt.show()

print("✓ SHAP figures saved")

In [ ]:
# ============================================================
# 9.3b.4  CONSISTENCIA INTER-MODELO DE RANKINGS SHAP
# ============================================================
# ¿Los modelos coinciden en qué features son importantes?
# Se calcula la correlación de Spearman entre rankings de importancia SHAP

shap_importance_rankings = {}
for name, sv in shap_values_dict.items():
    if isinstance(sv, list):
        sv = sv[0]
    # Mean absolute SHAP value por feature
    mean_abs = np.abs(sv).mean(axis=0)
    if len(mean_abs) == len(feature_cols):
        ranking = pd.Series(mean_abs, index=feature_cols).rank(ascending=False)
        shap_importance_rankings[name] = ranking

if len(shap_importance_rankings) >= 2:
    ranking_df = pd.DataFrame(shap_importance_rankings)
    corr_matrix = ranking_df.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='RdYlGn', center=0.5, vmin=-1, vmax=1,
                ax=ax, linewidths=0.5, square=True,
                cbar_kws={'label': 'Spearman ρ (ranking SHAP)'})
    ax.set_title('Consistencia Inter-Modelo: Correlación de Rankings SHAP',
                 fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_shap_ranking_correlation.png', bbox_inches='tight')
    plt.show()

    print("\nCorrelaciones medias de ranking SHAP por modelo:")
    mean_corr = corr_matrix.mean(axis=1).sort_values(ascending=False)
    for name, val in mean_corr.items():
        print(f"  {name:20s}: ρ̄ = {val:.3f}")
else:
    print("⚠️ Insuficientes modelos con SHAP para comparar rankings")

In [ ]:
shap_importance_rankings = {}
for name, sv in shap_values_dict.items():
    if isinstance(sv, list):
        sv = sv[0]
    mean_abs = np.abs(sv).mean(axis=0)
    if len(mean_abs) == len(feature_cols):
        ranking = pd.Series(mean_abs, index=feature_cols).rank(ascending=False)
        shap_importance_rankings[name] = ranking

if len(shap_importance_rankings) >= 2:
    ranking_df = pd.DataFrame(shap_importance_rankings)
    corr_matrix = ranking_df.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='RdYlGn', center=0.5, vmin=-1, vmax=1,
                ax=ax, linewidths=0.5, square=True,
                cbar_kws={'label': 'Spearman ρ (SHAP ranking)'})
    ax.set_title('Inter-Model Consistency: SHAP Ranking Correlation',
                 fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'english/fig_shap_ranking_correlation.png', bbox_inches='tight')
    plt.show()

    print("\nMean SHAP ranking correlations by model:")
    mean_corr = corr_matrix.mean(axis=1).sort_values(ascending=False)
    for name, val in mean_corr.items():
        print(f"  {name:20s}: ρ̄ = {val:.3f}")
else:
    print("⚠️ Insufficient models with SHAP to compare rankings")

#### 9.3c Análisis de Routing del MoE — t-SNE/UMAP de Asignación de Expertos

Se proyecta el espacio de features a 2D usando t-SNE y UMAP, coloreando cada punto por el experto asignado por el gating network. Esto revela las **regiones del espacio de entrada que cada experto "posee"** y valida que la especialización es semánticamente coherente.

In [ ]:
# ============================================================
# 9.3c  t-SNE / UMAP DE ASIGNACIÓN DE EXPERTOS
# ============================================================
from sklearn.manifold import TSNE

# Instalar UMAP si está disponible
try:
    import umap
    HAS_UMAP = True
except:
    HAS_UMAP = False
    print("⚠️ UMAP no disponible, se usará solo t-SNE")

if moe_path.exists() and hasattr(moe_model, 'predict_with_metadata'):
    print("Obteniendo metadatos de routing del MoE en test set...")
    moe_preds_meta, moe_metadata = moe_model.predict_with_metadata(df_test)

    print(f"  Observaciones: {len(moe_metadata)}")
    print(f"  Modos de gating: {moe_metadata['gating_mode'].value_counts().to_dict()}")
    print(f"  Expertos activos: {moe_metadata['expert_id'].nunique()}")

    # Submuestreo para t-SNE (máx 5000 puntos)
    N_TSNE = min(5000, len(moe_metadata))
    idx_sample = np.random.choice(len(moe_metadata), N_TSNE, replace=False)
    X_tsne_input = X_test.iloc[idx_sample].values
    meta_sample = moe_metadata.iloc[idx_sample].reset_index(drop=True)

    # Escalar para t-SNE
    X_tsne_scaled = StandardScaler().fit_transform(X_tsne_input)

    # --- t-SNE ---
    print("\nCalculando t-SNE (perplexity=30)...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED,
                n_iter=1000, learning_rate='auto', init='pca')
    X_2d_tsne = tsne.fit_transform(X_tsne_scaled)

    # --- UMAP ---
    if HAS_UMAP:
        print("Calculando UMAP (n_neighbors=15)...")
        reducer = umap.UMAP(n_components=2, n_neighbors=15,
                            min_dist=0.1, random_state=RANDOM_SEED)
        X_2d_umap = reducer.fit_transform(X_tsne_scaled)

    # --- Visualización ---
    n_plots = 2 if HAS_UMAP else 1
    fig, axes = plt.subplots(1, n_plots + 1, figsize=(7 * (n_plots + 1), 6))

    # Colores por experto
    expert_ids_unique = meta_sample['expert_id'].unique()
    expert_cmap = plt.cm.get_cmap('tab10', len(expert_ids_unique))
    expert_color_map = {eid: expert_cmap(i) for i, eid in enumerate(sorted(expert_ids_unique))}
    colors_expert = [expert_color_map[eid] for eid in meta_sample['expert_id']]

    # t-SNE coloreado por experto
    ax = axes[0]
    for eid in sorted(expert_ids_unique):
        mask = meta_sample['expert_id'] == eid
        ax.scatter(X_2d_tsne[mask, 0], X_2d_tsne[mask, 1],
                   c=[expert_color_map[eid]], s=8, alpha=0.6, label=eid)
    ax.set_title('t-SNE — Asignación de Expertos', fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=7, markerscale=3, title='Experto', loc='best')

    # UMAP coloreado por experto
    if HAS_UMAP:
        ax = axes[1]
        for eid in sorted(expert_ids_unique):
            mask = meta_sample['expert_id'] == eid
            ax.scatter(X_2d_umap[mask, 0], X_2d_umap[mask, 1],
                       c=[expert_color_map[eid]], s=8, alpha=0.6, label=eid)
        ax.set_title('UMAP — Asignación de Expertos', fontweight='bold')
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
        ax.legend(fontsize=7, markerscale=3, title='Experto', loc='best')

    # t-SNE coloreado por modo de gating (hard vs soft)
    ax = axes[-1]
    mode_colors = {'hard': '#2A9D8F', 'soft': '#E76F51', 'hard_fallback': '#E9C46A'}
    for mode, color in mode_colors.items():
        mask = meta_sample['gating_mode'] == mode
        if mask.sum() > 0:
            ax.scatter(X_2d_tsne[mask, 0], X_2d_tsne[mask, 1],
                       c=color, s=8, alpha=0.6, label=f'{mode} ({mask.sum():,})')
    ax.set_title('t-SNE — Modo de Gating', fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=9, markerscale=3)

    fig.suptitle('Visualización del Routing MoE en el Espacio de Features',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_routing_tsne_umap.png', bbox_inches='tight')
    plt.show()

    print("✓ Figuras t-SNE/UMAP guardadas")
else:
    print("⚠️ Modelo MoE no disponible o no soporta predict_with_metadata")

In [ ]:
n_plots = 2 if HAS_UMAP else 1
fig, axes = plt.subplots(1, n_plots + 1, figsize=(7 * (n_plots + 1), 6))

expert_ids_unique = meta_sample['expert_id'].unique()
expert_cmap = plt.cm.get_cmap('tab10', len(expert_ids_unique))
expert_color_map = {eid: expert_cmap(i) for i, eid in enumerate(sorted(expert_ids_unique))}
colors_expert = [expert_color_map[eid] for eid in meta_sample['expert_id']]

ax = axes[0]
for eid in sorted(expert_ids_unique):
    mask = meta_sample['expert_id'] == eid
    ax.scatter(X_2d_tsne[mask, 0], X_2d_tsne[mask, 1],
               c=[expert_color_map[eid]], s=8, alpha=0.6, label=eid)
ax.set_title('t-SNE — Expert Assignment', fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=7, markerscale=3, title='Expert', loc='best')

if HAS_UMAP:
    ax = axes[1]
    for eid in sorted(expert_ids_unique):
        mask = meta_sample['expert_id'] == eid
        ax.scatter(X_2d_umap[mask, 0], X_2d_umap[mask, 1],
                   c=[expert_color_map[eid]], s=8, alpha=0.6, label=eid)
    ax.set_title('UMAP — Expert Assignment', fontweight='bold')
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.legend(fontsize=7, markerscale=3, title='Expert', loc='best')

ax = axes[-1]
mode_colors = {'hard': '#2A9D8F', 'soft': '#E76F51', 'hard_fallback': '#E9C46A'}
for mode, color in mode_colors.items():
    mask = meta_sample['gating_mode'] == mode
    if mask.sum() > 0:
        ax.scatter(X_2d_tsne[mask, 0], X_2d_tsne[mask, 1],
                   c=color, s=8, alpha=0.6, label=f'{mode} ({mask.sum():,})')
ax.set_title('t-SNE — Gating Mode', fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=9, markerscale=3)

fig.suptitle('Visualization of MoE Routing in Feature Space',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_routing_tsne_umap.png', bbox_inches='tight')
plt.show()

print("✓ t-SNE/UMAP figures saved")

In [ ]:
# ============================================================
# 9.3c.2  ANÁLISIS CONDICIONAL: EXPERTO × METADATA DEL DOMINIO
# ============================================================
# ¿Qué experto se activa en cada contexto operativo?
# Cruce: experto asignado × hora del día, día de la semana, festivos

if moe_path.exists() and 'moe_metadata' in dir():
    # Añadir metadata temporal del test set
    analysis_df = moe_metadata.copy()
    analysis_df['hour'] = df_test.index[:len(analysis_df)].hour
    analysis_df['dayofweek'] = df_test.index[:len(analysis_df)].dayofweek
    analysis_df['confianza'] = analysis_df['min_conf']

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # 1. Distribución de expertos por hora del día
    ax = axes[0]
    expert_hour = pd.crosstab(analysis_df['hour'], analysis_df['expert_id'], normalize='index')
    expert_hour.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.85)
    ax.set_title('Activación de Expertos por Hora', fontweight='bold')
    ax.set_xlabel('Hora del Día')
    ax.set_ylabel('Proporción')
    ax.legend(title='Experto', fontsize=7, bbox_to_anchor=(1.02, 1))

    # 2. Distribución de expertos por día de la semana
    ax = axes[1]
    expert_dow = pd.crosstab(analysis_df['dayofweek'], analysis_df['expert_id'], normalize='index')
    dow_labels = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
    expert_dow.index = [dow_labels[i] for i in expert_dow.index]
    expert_dow.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.85)
    ax.set_title('Activación de Expertos por Día', fontweight='bold')
    ax.set_xlabel('Día de la Semana')
    ax.set_ylabel('Proporción')
    ax.legend(title='Experto', fontsize=7, bbox_to_anchor=(1.02, 1))

    # 3. Distribución de confianza del gating
    ax = axes[2]
    for mode in ['hard', 'soft', 'hard_fallback']:
        mask = analysis_df['gating_mode'] == mode
        if mask.sum() > 0:
            ax.hist(analysis_df.loc[mask, 'confianza'], bins=30, alpha=0.6,
                    label=f'{mode} (n={mask.sum():,})', density=True)
    ax.axvline(x=moe_model.gating_threshold, color='red', linestyle='--',
               linewidth=2, label=f'Threshold={moe_model.gating_threshold}')
    ax.set_title('Distribución de Confianza del Gating', fontweight='bold')
    ax.set_xlabel('Confianza Mínima (min_conf)')
    ax.set_ylabel('Densidad')
    ax.legend()

    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_expert_activation_analysis.png', bbox_inches='tight')
    plt.show()

    # Tabla resumen
    print("\n" + "="*80)
    print("  ANÁLISIS CONDICIONAL: EXPERTO × CONTEXTO OPERATIVO")
    print("="*80)
    print("\nExperto dominante por franja horaria:")
    for hour_range, label in [(range(0,6), 'Madrugada (0-5)'),
                               (range(6,12), 'Mañana (6-11)'),
                               (range(12,18), 'Tarde (12-17)'),
                               (range(18,24), 'Noche (18-23)')]:
        mask = analysis_df['hour'].isin(hour_range)
        dominant = analysis_df.loc[mask, 'expert_id'].mode()
        pct = (analysis_df.loc[mask, 'expert_id'] == dominant.iloc[0]).mean() * 100 if len(dominant) > 0 else 0
        print(f"  {label:20s} → {dominant.iloc[0] if len(dominant) > 0 else 'N/A'} ({pct:.1f}%)")

    print("\nExperto dominante por día:")
    for dow, label in enumerate(dow_labels):
        mask = analysis_df['dayofweek'] == dow
        dominant = analysis_df.loc[mask, 'expert_id'].mode()
        pct = (analysis_df.loc[mask, 'expert_id'] == dominant.iloc[0]).mean() * 100 if len(dominant) > 0 else 0
        print(f"  {label:5s} → {dominant.iloc[0] if len(dominant) > 0 else 'N/A'} ({pct:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
expert_hour = pd.crosstab(analysis_df['hour'], analysis_df['expert_id'], normalize='index')
expert_hour.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.85)
ax.set_title('Expert Activation by Hour', fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Proportion')
ax.legend(title='Expert', fontsize=7, bbox_to_anchor=(1.02, 1))

ax = axes[1]
expert_dow = pd.crosstab(analysis_df['dayofweek'], analysis_df['expert_id'], normalize='index')
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
expert_dow.index = [dow_labels[i] for i in expert_dow.index]
expert_dow.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', width=0.85)
ax.set_title('Expert Activation by Day', fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Proportion')
ax.legend(title='Expert', fontsize=7, bbox_to_anchor=(1.02, 1))

ax = axes[2]
for mode in ['hard', 'soft', 'hard_fallback']:
    mask = analysis_df['gating_mode'] == mode
    if mask.sum() > 0:
        ax.hist(analysis_df.loc[mask, 'confianza'], bins=30, alpha=0.6,
                label=f'{mode} (n={mask.sum():,})', density=True)
ax.axvline(x=moe_model.gating_threshold, color='red', linestyle='--',
           linewidth=2, label=f'Threshold={moe_model.gating_threshold}')
ax.set_title('Gating Confidence Distribution', fontweight='bold')
ax.set_xlabel('Minimum Confidence (min_conf)')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_expert_activation_analysis.png', bbox_inches='tight')
plt.show()

#### 9.3d Estabilidad del Routing

Para muestras similares en el espacio de features, ¿el router asigna consistentemente el mismo experto? Alta inestabilidad (muestras cercanas van a expertos diferentes) indica fronteras de decisión frágiles.

**Métrica**: Para cada muestra, se buscan sus $k=10$ vecinos más cercanos y se calcula la fracción que fue asignada al mismo experto. Un routing estable debería tener consistencia > 80%.

In [ ]:
# ============================================================
# 10.3d  ESTABILIDAD DEL ROUTING
# ============================================================
from sklearn.neighbors import NearestNeighbors

if moe_path.exists() and 'moe_metadata' in dir():
    print("Calculando estabilidad del routing...")

    # Usar subset para eficiencia
    N_STABILITY = min(3000, len(moe_metadata))
    idx_stab = np.random.choice(len(moe_metadata), N_STABILITY, replace=False)
    X_stab = StandardScaler().fit_transform(X_test.iloc[idx_stab].values)
    experts_stab = moe_metadata.iloc[idx_stab]['expert_id'].values

    # k-NN
    K_NEIGHBORS = 10
    nn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric='euclidean')
    nn_model.fit(X_stab)
    distances, indices = nn_model.kneighbors(X_stab)

    # Para cada muestra, fracción de vecinos con el mismo experto
    stability_scores = []
    for i in range(N_STABILITY):
        neighbors = indices[i, 1:]  # Excluir self
        same_expert = np.mean(experts_stab[neighbors] == experts_stab[i])
        stability_scores.append(same_expert)

    stability_scores = np.array(stability_scores)

    print(f"\n  Estabilidad del Routing (k={K_NEIGHBORS} vecinos):")
    print(f"    Media:   {stability_scores.mean():.4f} ({stability_scores.mean()*100:.1f}%)")
    print(f"    Mediana: {np.median(stability_scores):.4f}")
    print(f"    Std:     {stability_scores.std():.4f}")
    print(f"    Min:     {stability_scores.min():.4f}")
    print(f"    >80%:    {(stability_scores > 0.8).mean()*100:.1f}% de las muestras")
    print(f"    >90%:    {(stability_scores > 0.9).mean()*100:.1f}% de las muestras")
    print(f"    =100%:   {(stability_scores == 1.0).mean()*100:.1f}% de las muestras")

    # Visualización
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Histograma de estabilidad
    ax = axes[0]
    ax.hist(stability_scores, bins=K_NEIGHBORS + 1, edgecolor='black', alpha=0.7, color='#2A9D8F')
    ax.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='Umbral 80%')
    ax.axvline(x=stability_scores.mean(), color='orange', linestyle='-',
               linewidth=2, label=f'Media={stability_scores.mean():.2f}')
    ax.set_xlabel(f'Fracción de vecinos (k={K_NEIGHBORS}) con mismo experto')
    ax.set_ylabel('Frecuencia')
    ax.set_title('Distribución de Estabilidad del Routing', fontweight='bold')
    ax.legend()

    # 2. Estabilidad por experto
    ax = axes[1]
    stab_df = pd.DataFrame({
        'expert_id': experts_stab,
        'stability': stability_scores
    })
    expert_stab = stab_df.groupby('expert_id')['stability'].agg(['mean', 'std', 'count'])
    expert_stab = expert_stab.sort_values('mean', ascending=True)
    bars = ax.barh(expert_stab.index, expert_stab['mean'],
                   xerr=expert_stab['std'], color='#457B9D',
                   edgecolor='white', capsize=3)
    ax.axvline(x=0.8, color='red', linestyle='--', alpha=0.7)
    ax.set_xlabel('Estabilidad Media')
    ax.set_title('Estabilidad por Experto', fontweight='bold')

    xmax = (expert_stab['mean'] + expert_stab['std']).max()
    ax.set_xlim(0, xmax + 0.20)

    for bar, (_, row) in zip(bars, expert_stab.iterrows()):
        ax.text(row['mean'] + row['std'] + 0.01, bar.get_y() + bar.get_height()/2,
                f'{row["mean"]:.2f} (n={int(row["count"])})', va='center', fontsize=8)

    # 3. t-SNE coloreado por estabilidad
    ax = axes[2]
    # Reusar t-SNE del bloque anterior si hay intersección
    stab_for_tsne = stability_scores[:min(N_STABILITY, N_TSNE)] if 'X_2d_tsne' in dir() else stability_scores
    if 'X_2d_tsne' in dir() and len(X_2d_tsne) >= len(stab_for_tsne):
        sc = ax.scatter(X_2d_tsne[:len(stab_for_tsne), 0],
                        X_2d_tsne[:len(stab_for_tsne), 1],
                        c=stab_for_tsne, cmap='RdYlGn', s=5, alpha=0.6)
        plt.colorbar(sc, ax=ax, label='Estabilidad')
        ax.set_title('t-SNE — Estabilidad del Routing', fontweight='bold')
        ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    else:
        # Calcular t-SNE local si no existe
        tsne_stab = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED).fit_transform(
            X_stab[:min(2000, len(X_stab))])
        sc = ax.scatter(tsne_stab[:, 0], tsne_stab[:, 1],
                        c=stability_scores[:len(tsne_stab)], cmap='RdYlGn', s=5, alpha=0.6)
        plt.colorbar(sc, ax=ax, label='Estabilidad')
        ax.set_title('t-SNE — Estabilidad del Routing', fontweight='bold')

    fig.suptitle(f'Análisis de Estabilidad del Routing MoE (k={K_NEIGHBORS})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_routing_stability.png', bbox_inches='tight')
    plt.show()

    # Guardar métricas
    stability_summary = {
        'k_neighbors': K_NEIGHBORS,
        'mean_stability': float(stability_scores.mean()),
        'median_stability': float(np.median(stability_scores)),
        'std_stability': float(stability_scores.std()),
        'pct_above_80': float((stability_scores > 0.8).mean()),
        'pct_above_90': float((stability_scores > 0.9).mean()),
        'pct_perfect': float((stability_scores == 1.0).mean()),
        'per_expert': expert_stab['mean'].to_dict(),
    }
    with open(BENCH_DIR / 'routing_stability.json', 'w') as f:
        json.dump(stability_summary, f, indent=2, default=str)

    print("\n✓ Análisis de estabilidad guardado")
else:
    print("⚠️ Modelo MoE no disponible para análisis de estabilidad")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.hist(stability_scores, bins=K_NEIGHBORS + 1, edgecolor='black', alpha=0.7, color='#2A9D8F')
ax.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='80% Threshold')
ax.axvline(x=stability_scores.mean(), color='orange', linestyle='-',
           linewidth=2, label=f'Mean={stability_scores.mean():.2f}')
ax.set_xlabel(f'Fraction of neighbors (k={K_NEIGHBORS}) with same expert')
ax.set_ylabel('Frequency')
ax.set_title('Routing Stability Distribution', fontweight='bold')
ax.legend()

ax = axes[1]
stab_df = pd.DataFrame({
    'expert_id': experts_stab,
    'stability': stability_scores
})
expert_stab = stab_df.groupby('expert_id')['stability'].agg(['mean', 'std', 'count'])
expert_stab = expert_stab.sort_values('mean', ascending=True)

bars = ax.barh(
    expert_stab.index,
    expert_stab['mean'],
    xerr=expert_stab['std'],
    color='#457B9D',
    edgecolor='white',
    capsize=3
)

ax.axvline(x=0.8, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('Mean Stability')
ax.set_title('Stability by Expert', fontweight='bold')

xmax = (expert_stab['mean'] + expert_stab['std']).max()
ax.set_xlim(0, xmax + 0.20)

for bar, (_, row) in zip(bars, expert_stab.iterrows()):
    label_x = row['mean'] + row['std'] + 0.01
    ax.text(
        label_x,
        bar.get_y() + bar.get_height() / 2,
        f'{row["mean"]:.2f} (n={int(row["count"])})',
        va='center',
        fontsize=8
    )

ax = axes[2]
stab_for_tsne = stability_scores[:min(N_STABILITY, N_TSNE)] if 'X_2d_tsne' in dir() else stability_scores
if 'X_2d_tsne' in dir() and len(X_2d_tsne) >= len(stab_for_tsne):
    sc = ax.scatter(X_2d_tsne[:len(stab_for_tsne), 0],
                    X_2d_tsne[:len(stab_for_tsne), 1],
                    c=stab_for_tsne, cmap='RdYlGn', s=5, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Stability')
    ax.set_title('t-SNE — Routing Stability', fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
else:
    tsne_stab = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED).fit_transform(
        X_stab[:min(2000, len(X_stab))]
    )
    sc = ax.scatter(tsne_stab[:, 0], tsne_stab[:, 1],
                    c=stability_scores[:len(tsne_stab)], cmap='RdYlGn', s=5, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='Stability')
    ax.set_title('t-SNE — Routing Stability', fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')

fig.suptitle(f'MoE Routing Stability Analysis (k={K_NEIGHBORS})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_routing_stability.png', bbox_inches='tight')
plt.show()

#### 9.3e Resumen Consolidado de Interpretabilidad

In [ ]:
# ============================================================
# 9.3e  RESUMEN CONSOLIDADO DE INTERPRETABILIDAD
# ============================================================
print("\n" + "=" * 100)
print("  RESUMEN CONSOLIDADO — ANÁLISIS DE INTERPRETABILIDAD")
print("=" * 100)

print("\n  1. ESCALA CUANTITATIVA (1-5):")
for _, row in interp_score_df.iterrows():
    print(f"     {row['Modelo']:30s} → {int(row['Score (1-5)'])}/5  ({row['Justificación']})")

print("\n  2. SHAP VALUES:")
print(f"     Modelos analizados: {len(shap_values_dict)}")
if shap_values_dict:
    # Top-3 features más importantes (promedio across models)
    all_importances = {}
    for name, sv in shap_values_dict.items():
        if isinstance(sv, list): sv = sv[0]
        if sv.shape[1] == len(feature_cols):
            for i, feat in enumerate(feature_cols):
                if feat not in all_importances:
                    all_importances[feat] = []
                all_importances[feat].append(np.abs(sv[:, i]).mean())
    if all_importances:
        avg_imp = {k: np.mean(v) for k, v in all_importances.items()}
        top5_global = sorted(avg_imp.items(), key=lambda x: -x[1])[:5]
        print("     Top-5 features más importantes (promedio across modelos):")
        for feat, imp in top5_global:
            print(f"       {feat:30s}: {imp:.4f}")

if moe_path.exists() and 'stability_scores' in dir():
    print(f"\n  3. ROUTING MoE:")
    print(f"     Expertos activos: {moe_metadata['expert_id'].nunique()}")
    print(f"     Hard gating: {(moe_metadata['gating_mode']=='hard').mean()*100:.1f}%")
    print(f"     Soft gating: {(moe_metadata['gating_mode']=='soft').mean()*100:.1f}%")
    print(f"\n  4. ESTABILIDAD DEL ROUTING:")
    print(f"     Media: {stability_scores.mean()*100:.1f}%")
    print(f"     Muestras con >80% consistencia: {(stability_scores > 0.8).mean()*100:.1f}%")

print("\n  5. CONCLUSIÓN:")
print("     El MoE ofrece un balance único: interpretabilidad operativa (score 4/5)")
print("     gracias al gating explícito por tipo de día/semana, con la capacidad")
print("     predictiva de modelos especializados por dominio.")
print("=" * 100)

# Guardar todo
interp_score_df.to_csv(BENCH_DIR / 'interpretability_scores.csv', index=False)
print(f"\n✓ Todos los archivos de interpretabilidad guardados en {BENCH_DIR}")

### 9.4 Pruebas Estadísticas

Se valida si las diferencias entre modelos son estadísticamente significativas usando:
1. **ANOVA de un factor** — H₀: todos los modelos tienen el mismo error medio
2. **Friedman test** — Alternativa no paramétrica para datos pareados
3. **Pruebas post-hoc** — Tukey HSD para identificar qué pares difieren significativamente
4. **Nemenyi test** — Post-hoc no paramétrico

**Metodología**: Se calcula el error absoluto por observación para cada modelo, generando distribuciones comparables.

> Referencia: Demšar (2006). *Statistical Comparisons of Classifiers over Multiple Datasets*. JMLR.

In [ ]:
# ============================================================
# 9.4a  PREPARACIÓN DE ERRORES POR OBSERVACIÓN
# ============================================================
# Para pruebas estadísticas necesitamos errores individuales por timestep
errors_dict = {}

# MoE
if moe_preds_test is not None:
    errors_dict['MoE'] = np.abs(y_test.values - moe_preds_test)

# Modelos feature-based
for name, r in hpo_results.items():
    if r['type'] == 'feature_based':
        model = r['model']
        preds = model.predict(X_test)
        errors_dict[name] = np.abs(y_test.values - preds)

# Modelos DL
for name, r in dl_hpo_results.items():
    model = r['model']
    X_test_sc_dl = scaler_X.transform(X_test)
    y_test_sc_dl = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()
    test_ds_dl = TimeSeriesDataset(X_test_sc_dl, y_test_sc_dl, WINDOW_SIZE)
    test_ld = DataLoader(test_ds_dl, batch_size=256, shuffle=False)
    preds = predict_dl(model, test_ld, scaler_y)
    # Alinear longitudes
    n_preds = min(len(preds), len(y_test) - WINDOW_SIZE)
    errors_dict[name] = np.abs(y_test.values[WINDOW_SIZE:WINDOW_SIZE+n_preds] - preds[:n_preds])

# Alinear al mínimo tamaño
min_len = min(len(e) for e in errors_dict.values()) if errors_dict else 0
for k in errors_dict:
    errors_dict[k] = errors_dict[k][:min_len]

errors_df = pd.DataFrame(errors_dict)
print(f"Errores absolutos calculados: {errors_df.shape[0]} observaciones × {errors_df.shape[1]} modelos")
print(f"Modelos: {list(errors_dict.keys())}")

In [ ]:
# ============================================================
# 9.4b  ANOVA DE UN FACTOR
# ============================================================
print("="*80)
print("  PRUEBA ANOVA DE UN FACTOR (One-Way ANOVA)")
print("="*80)
print("  H₀: Los errores medios de todos los modelos son iguales")
print("  H₁: Al menos un modelo tiene un error medio diferente\n")

groups = [errors_dict[k] for k in errors_dict]
f_stat, p_value_anova = sp_stats.f_oneway(*groups)

print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value:     {p_value_anova:.2e}")
print(f"  Significativo (α=0.05): {'✓ SÍ' if p_value_anova < 0.05 else '✗ NO'}")
print(f"  Significativo (α=0.01): {'✓ SÍ' if p_value_anova < 0.01 else '✗ NO'}")

# Verificar supuestos
# Normalidad (Shapiro para submuestras)
print("\n  Verificación de supuestos:")
for name, errors in errors_dict.items():
    sample = np.random.choice(errors, min(500, len(errors)), replace=False)
    _, p_shapiro = sp_stats.shapiro(sample)
    print(f"    {name:20s} — Shapiro-Wilk p={p_shapiro:.4f} "
          f"({'Normal' if p_shapiro > 0.05 else 'No normal'})")

# Levene (homogeneidad de varianzas)
_, p_levene = sp_stats.levene(*groups)
print(f"\n    Levene (homocedasticidad): p={p_levene:.4f} "
      f"({'Homogéneo' if p_levene > 0.05 else 'Heterogéneo'})")

In [ ]:
# ============================================================
# 9.4c  PRUEBA DE FRIEDMAN (no paramétrica)
# ============================================================
print("\n" + "="*80)
print("  PRUEBA DE FRIEDMAN (alternativa no paramétrica)")
print("="*80)
print("  H₀: No hay diferencia en el ranking de modelos entre observaciones")
print("  H₁: Al menos un modelo tiene un ranking diferente\n")

# Para Friedman necesitamos muestras de igual tamaño (ya alineadas)
try:
    chi2_stat, p_value_friedman = sp_stats.friedmanchisquare(*groups)
    print(f"  χ² statistic: {chi2_stat:.4f}")
    print(f"  p-value:      {p_value_friedman:.2e}")
    print(f"  Significativo (α=0.05): {'✓ SÍ' if p_value_friedman < 0.05 else '✗ NO'}")
except Exception as e:
    print(f"  ⚠️ Friedman falló: {e}")
    p_value_friedman = 1.0

In [ ]:
# ============================================================
# 9.4d  PRUEBA POST-HOC — TUKEY HSD
# ============================================================
from itertools import combinations

print("\n" + "="*80)
print("  PRUEBAS POST-HOC — Tukey HSD (comparaciones por pares)")
print("="*80)

if p_value_anova < 0.05:
    print("  ANOVA significativo → procedemos con comparaciones por pares\n")

    # Tukey HSD manual usando statsmodels
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

    all_errors = []
    all_labels = []
    for name, errors in errors_dict.items():
        all_errors.extend(errors)
        all_labels.extend([name] * len(errors))

    tukey = pairwise_tukeyhsd(all_errors, all_labels, alpha=0.05)
    print(tukey.summary())

    # Guardar resultados
    tukey_df = pd.DataFrame(data=tukey._results_table.data[1:],
                            columns=tukey._results_table.data[0])
    tukey_df.to_csv(BENCH_DIR / 'tukey_hsd_results.csv', index=False)

    # Visualización de intervalos de confianza
    fig, ax = plt.subplots(figsize=(14, 8))
    tukey.plot_simultaneous(ax=ax)
    ax.set_title('Tukey HSD — Intervalos de Confianza Simultáneos', fontweight='bold')
    ax.set_xlabel('Error Absoluto Medio')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_tukey_hsd.png', bbox_inches='tight')
    plt.show()
else:
    print("  ANOVA no significativo → no se requieren pruebas post-hoc")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
tukey.plot_simultaneous(ax=ax)
ax.set_title('Tukey HSD — Simultaneous Confidence Intervals', fontweight='bold')
ax.set_xlabel('Mean Absolute Error')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_tukey_hsd.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 9.4e  NEMENYI POST-HOC (no paramétrico)
# ============================================================
print("\n" + "="*80)
print("  POST-HOC NO PARAMÉTRICO — Nemenyi / Wilcoxon por pares")
print("="*80)

if p_value_friedman < 0.05:
    print("  Friedman significativo → comparaciones por pares con Wilcoxon\n")

    model_names = list(errors_dict.keys())
    n_models = len(model_names)
    pairwise_pvalues = np.ones((n_models, n_models))

    for i, j in combinations(range(n_models), 2):
        _, p_val = sp_stats.wilcoxon(
            errors_dict[model_names[i]],
            errors_dict[model_names[j]],
            alternative='two-sided'
        )
        # Corrección de Bonferroni
        n_comparisons = n_models * (n_models - 1) / 2
        p_corrected = min(p_val * n_comparisons, 1.0)
        pairwise_pvalues[i, j] = p_corrected
        pairwise_pvalues[j, i] = p_corrected

    # Heatmap de p-valores
    pairwise_df = pd.DataFrame(pairwise_pvalues, index=model_names, columns=model_names)

    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(pairwise_pvalues, dtype=bool))
    sns.heatmap(pairwise_df, mask=mask, annot=True, fmt='.4f',
                cmap='RdYlGn', center=0.05, vmin=0, vmax=1,
                ax=ax, linewidths=0.5,
                cbar_kws={'label': 'p-valor corregido (Bonferroni)'})
    ax.set_title('Wilcoxon por Pares (corregido Bonferroni)', fontweight='bold')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_pairwise_wilcoxon.png', bbox_inches='tight')
    plt.show()

    pairwise_df.to_csv(BENCH_DIR / 'pairwise_wilcoxon_pvalues.csv')
    print("✓ Resultados guardados")
else:
    print("  Friedman no significativo → no se requieren comparaciones post-hoc")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(pairwise_pvalues, dtype=bool))
sns.heatmap(pairwise_df, mask=mask, annot=True, fmt='.4f',
            cmap='RdYlGn', center=0.05, vmin=0, vmax=1,
            ax=ax, linewidths=0.5,
            cbar_kws={'label': 'corrected p-value (Bonferroni)'})
ax.set_title('Pairwise Wilcoxon Test (Bonferroni-corrected)', fontweight='bold')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_pairwise_wilcoxon.png', bbox_inches='tight')
plt.show()
print("✓ Results saved")


In [ ]:
# ============================================================
# 9.4f  DISTRIBUCIÓN DE ERRORES — BOXPLOT COMPARATIVO
# ============================================================
fig, ax = plt.subplots(figsize=(16, 8))

# Ordenar modelos por mediana de error
medians = errors_df.median().sort_values()
errors_ordered = errors_df[medians.index]

bp = ax.boxplot([errors_ordered[col].values for col in errors_ordered.columns],
                labels=errors_ordered.columns,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=2))

# Colorear por familia
for i, col in enumerate(errors_ordered.columns):
    family = final_results.get(f'{col} (Opt)', final_results.get(f'{col} (Opt)',
             final_results.get(f'{col} (Prop)', {})))
    fam = family.get('family', 'Unknown') if isinstance(family, dict) else 'Unknown'
    bp['boxes'][i].set_facecolor(family_colors.get(fam, '#CCCCCC'))
    bp['boxes'][i].set_alpha(0.7)

ax.set_ylabel('Error Absoluto (kWh)')
ax.set_title('Distribución de Errores Absolutos por Modelo', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

patches = [mpatches.Patch(color=c, label=f, alpha=0.7) for f, c in family_colors.items()]
ax.legend(handles=patches, loc='upper right')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'fig_error_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

medians = errors_df.median().sort_values()
errors_ordered = errors_df[medians.index]

bp = ax.boxplot([errors_ordered[col].values for col in errors_ordered.columns],
                labels=errors_ordered.columns,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=2))

for i, col in enumerate(errors_ordered.columns):
    family = final_results.get(f'{col} (Opt)', final_results.get(f'{col} (Opt)',
             final_results.get(f'{col} (Prop)', {})))
    fam = family.get('family', 'Unknown') if isinstance(family, dict) else 'Unknown'
    bp['boxes'][i].set_facecolor(family_colors.get(fam, '#CCCCCC'))
    bp['boxes'][i].set_alpha(0.7)

ax.set_ylabel('Absolute Error (kWh)')
ax.set_title('Distribution of Absolute Errors by Model', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

patches = [mpatches.Patch(color=c, label=f, alpha=0.7) for f, c in family_colors.items()]
ax.legend(handles=patches, loc='upper right')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'english/fig_error_distributions.png', bbox_inches='tight')
plt.show()

---
### 9.5 Prueba de Robustez: Comportamiento Atípico COVID-19 (2020-2021)

Los datos de 2020 y 2021 fueron **excluidos deliberadamente del entrenamiento** del MoE y de todos los modelos del benchmark debido al impacto disruptivo del COVID-19 en los patrones de demanda eléctrica. Según Buechler et al. (2020), la pandemia provocó caídas de hasta 26% en la demanda diaria en algunos países, generando comportamientos sin precedente en las series de tiempo.

Este análisis utiliza esos datos excluidos como **test de estrés out-of-distribution (OOD)**: se evalúa cómo se comportan los modelos entrenados en datos "normales" (2022-2025) cuando se enfrentan a patrones que nunca vieron.

#### Momentos críticos seleccionados para Colombia:

| # | Período | Evento | Impacto esperado en demanda |
|---|---------|--------|-----------------------------|
| 1 | **Ene 6 – Mar 15, 2020** | Pre-COVID (baseline normal) | Referencia: patrón típico antes del choque |
| 2 | **Mar 25 – Abr 30, 2020** | Cuarentena estricta nacional (Decreto 457) | Caída masiva: cierre industrial, comercial y restricción de movilidad |
| 3 | **Abr 6 – Abr 12, 2020** | Semana Santa bajo confinamiento total | Demanda mínima histórica: festivo + lockdown simultáneo |
| 4 | **Jun 1 – Ago 31, 2020** | Reapertura gradual y selectiva | Recuperación oscilante: sectores reabren a ritmos distintos |
| 5 | **Dic 15, 2020 – Ene 15, 2021** | Temporada navideña pandémica | Festivos atípicos: restricciones parciales + nueva ola |
| 6 | **Abr 28 – May 31, 2021** | Paro Nacional (estallido social) | Disrupción económica: bloqueos, toques de queda sectoriales |

> **Nota**: Ninguno de estos datos fue usado para entrenamiento, validación ni test de los modelos.

In [ ]:
# ============================================================
# 9.5.0  CARGA DE DATOS COMPLETOS (INCLUYENDO 2020-2021)
# ============================================================
# El features.parquet contiene TODO el histórico desde 2020
# Necesitamos reconstruir las features para 2020-2021

print("=" * 80)
print("  CARGA DE DATOS COVID (2020-2021) — Excluidos del entrenamiento")
print("=" * 80)

# Recargar dataset completo
features_path_full = Path(DATA_DIR) / 'processed' / 'features.parquet'

if features_path_full.exists():
    df_full = pd.read_parquet(features_path_full)
    if not isinstance(df_full.index, pd.DatetimeIndex):
        for col in ['timestamp', 'datetime', 'date']:
            if col in df_full.columns:
                df_full = df_full.set_index(col)
                break
    df_full = df_full.sort_index()

    # Filtrar solo 2020-2021
    df_covid_era = df_full.loc['2020-01-01':'2021-12-31'].copy()
    print(f"  Datos 2020-2021: {len(df_covid_era):,} filas")
    print(f"  Rango: {df_covid_era.index.min()} → {df_covid_era.index.max()}")

    # Verificar que las features del modelo existan
    missing_feats = [c for c in feature_cols if c not in df_covid_era.columns]
    if missing_feats:
        print(f"  ⚠️ Features faltantes en datos COVID: {missing_feats[:10]}...")
        # Intentar reconstruir features básicas
        available_feats = [c for c in feature_cols if c in df_covid_era.columns]
        print(f"  Features disponibles: {len(available_feats)}/{len(feature_cols)}")
    else:
        available_feats = feature_cols
        print(f"  ✓ Todas las {len(feature_cols)} features disponibles")

    # Limpiar NaN
    df_covid_clean = df_covid_era[[TARGET] + available_feats].dropna()
    print(f"  Filas limpias: {len(df_covid_clean):,}")

    # Verificar demanda
    demand_covid = df_covid_clean[TARGET]
    print(f"\n  Estadísticas de demanda 2020-2021:")
    print(f"    Media:  {demand_covid.mean():,.0f} kWh")
    print(f"    Min:    {demand_covid.min():,.0f} kWh")
    print(f"    Max:    {demand_covid.max():,.0f} kWh")
    print(f"    Std:    {demand_covid.std():,.0f} kWh")

    # Comparar con datos de entrenamiento (2022-2025)
    demand_train = y_train
    print(f"\n  Comparación con datos de entrenamiento (2022-2025):")
    print(f"    Media train: {demand_train.mean():,.0f} kWh")
    print(f"    Media COVID: {demand_covid.mean():,.0f} kWh")
    pct_diff = (demand_covid.mean() - demand_train.mean()) / demand_train.mean() * 100
    print(f"    Diferencia:  {pct_diff:+.1f}%")

else:
    print("⚠️ features.parquet no encontrado")
    print("   Generando datos sintéticos de COVID para demostración...")

    # Datos sintéticos que simulan el impacto COVID
    np.random.seed(RANDOM_SEED)
    idx_covid = pd.date_range('2020-01-01', '2021-12-31', freq='h')
    n_covid = len(idx_covid)

    # Demanda base + estacionalidad
    base = (7000 + 800 * np.sin(2*np.pi*np.arange(n_covid)/24)
            + 300 * np.sin(2*np.pi*np.arange(n_covid)/168)
            + np.random.normal(0, 150, n_covid))

    # Simular caída COVID (marzo-mayo 2020)
    covid_mask = (idx_covid >= '2020-03-25') & (idx_covid <= '2020-05-31')
    base[covid_mask] *= 0.78  # caída ~22%

    # Simular recuperación gradual (junio-dic 2020)
    recovery_mask = (idx_covid >= '2020-06-01') & (idx_covid <= '2020-12-31')
    recovery_factor = np.linspace(0.82, 0.95, recovery_mask.sum())
    base[recovery_mask] *= recovery_factor

    # Simular paro nacional (abril-mayo 2021)
    paro_mask = (idx_covid >= '2021-04-28') & (idx_covid <= '2021-05-31')
    base[paro_mask] *= 0.88

    base = base.clip(4000, 12000)
    df_covid_clean = pd.DataFrame({TARGET: base}, index=idx_covid)
    df_covid_clean.index.name = 'timestamp'

    # Generar features sintéticas
    df_covid_clean['hour'] = df_covid_clean.index.hour
    df_covid_clean['dayofweek'] = df_covid_clean.index.dayofweek
    df_covid_clean['month'] = df_covid_clean.index.month
    df_covid_clean['hour_sin']  = np.sin(2*np.pi*df_covid_clean['hour']/24)
    df_covid_clean['hour_cos']  = np.cos(2*np.pi*df_covid_clean['hour']/24)
    df_covid_clean['dow_sin']   = np.sin(2*np.pi*df_covid_clean['dayofweek']/7)
    df_covid_clean['dow_cos']   = np.cos(2*np.pi*df_covid_clean['dayofweek']/7)
    df_covid_clean['month_sin'] = np.sin(2*np.pi*df_covid_clean['month']/12)
    df_covid_clean['month_cos'] = np.cos(2*np.pi*df_covid_clean['month']/12)
    df_covid_clean['is_holiday'] = 0
    df_covid_clean['is_bridge']  = 0
    for lag in [1,2,3,23,24,25,47,48,49,167,168,169]:
        df_covid_clean[f'demand_lag_{lag}'] = df_covid_clean[TARGET].shift(lag)
    for w in [6,12,24,48,168]:
        df_covid_clean[f'demand_rmean_{w}'] = df_covid_clean[TARGET].shift(1).rolling(w).mean()
        df_covid_clean[f'demand_rstd_{w}']  = df_covid_clean[TARGET].shift(1).rolling(w).std()
    df_covid_clean.dropna(inplace=True)
    available_feats = [c for c in feature_cols if c in df_covid_clean.columns]
    print(f"  Datos sintéticos COVID: {len(df_covid_clean):,} filas")

#### 9.5.1 Definición de Períodos Críticos

In [ ]:
# ============================================================
# 9.5.1  DEFINICIÓN DE PERÍODOS CRÍTICOS
# ============================================================

COVID_PERIODS = {
    'P1_PreCOVID': {
        'start': '2020-01-06', 'end': '2020-03-15',
        'label': 'Pre-COVID\n(Baseline)',
        'description': 'Patrón normal antes del choque pandémico',
        'color': '#2A9D8F',
        'expected': 'Referencia: los modelos deberían funcionar razonablemente bien'
    },
    'P2_Cuarentena': {
        'start': '2020-03-25', 'end': '2020-04-30',
        'label': 'Cuarentena\nEstricta',
        'description': 'Decreto 457: confinamiento total nacional',
        'color': '#E63946',
        'expected': 'Caída masiva de demanda (~15-25%). Máximo estrés para los modelos'
    },
    'P3_SemanaSanta': {
        'start': '2020-04-04', 'end': '2020-04-12',
        'label': 'Semana Santa\n+ Lockdown',
        'description': 'Festivo religioso durante confinamiento total',
        'color': '#D62828',
        'expected': 'Demanda mínima histórica: festivo + lockdown simultáneo'
    },
    'P4_Reapertura': {
        'start': '2020-06-01', 'end': '2020-08-31',
        'label': 'Reapertura\nGradual',
        'description': 'Sectores reabren con protocolos, demanda oscilante',
        'color': '#F4A261',
        'expected': 'Recuperación parcial. Modelos deberían mejorar vs P2'
    },
    'P5_NavidadCOVID': {
        'start': '2020-12-15', 'end': '2021-01-15',
        'label': 'Navidad\nPandémica',
        'description': 'Temporada festiva con restricciones parciales + nueva ola',
        'color': '#E9C46A',
        'expected': 'Festivos atípicos: ni patrón normal ni lockdown total'
    },
    'P6_ParoNacional': {
        'start': '2021-04-28', 'end': '2021-05-31',
        'label': 'Paro\nNacional',
        'description': 'Estallido social: bloqueos viales, toques de queda sectoriales',
        'color': '#264653',
        'expected': 'Disrupción económica localizada. Demanda irregular'
    },
}

# Validar disponibilidad de datos
print("Períodos críticos COVID seleccionados:")
print("-" * 90)
for pid, info in COVID_PERIODS.items():
    mask = (df_covid_clean.index >= info['start']) & (df_covid_clean.index <= info['end'])
    n_hours = mask.sum()
    if n_hours > 0:
        mean_demand = df_covid_clean.loc[mask, TARGET].mean()
        print(f"  {pid:20s} | {info['start']} → {info['end']} | "
              f"{n_hours:5,}h | Demanda media: {mean_demand:,.0f} kWh")
        info['n_hours'] = n_hours
        info['available'] = True
    else:
        print(f"  {pid:20s} | {info['start']} → {info['end']} | ⚠️ SIN DATOS")
        info['available'] = False

In [ ]:
torch.cuda.empty_cache()

#### 9.5.2 Predicción por Período — MoE vs. Mejores Modelos

In [ ]:
# ============================================================
# 9.5.2  PREDICCIONES EN PERÍODOS COVID
# ============================================================

# Recopilar los mejores modelos de cada familia (optimizados)
covid_models = OrderedDict()

# MoE
if moe_path.exists():
    covid_models['MoE (Propuesto)'] = {
        'model': moe_model,
        'family': 'MoE',
        'predict_fn': lambda m, X_df: m.predict(X_df),
        'needs_df': True,  # MoE necesita DataFrame completo
    }

# Mejor estadístico feature-based
for name in list(top3_stat.keys()):
    if name in hpo_results and hpo_results[name]['type'] == 'feature_based':
        covid_models[f'{name} (Estadístico)'] = {
            'model': hpo_results[name]['model'],
            'family': 'Estadístico',
            'predict_fn': lambda m, X: m.predict(X),
            'needs_df': False,
        }
        break

# Mejor ML
for name in list(top3_ml.keys()):
    if name in hpo_results and hpo_results[name]['type'] == 'feature_based':
        covid_models[f'{name} (Machine Learning)'] = {
            'model': hpo_results[name]['model'],
            'family': 'ML',
            'predict_fn': lambda m, X: m.predict(X),
            'needs_df': False,
        }
        break

# Mejor DL
for name in list(top3_dl.keys()):
    if name in dl_hpo_results:
        dl_model_covid = dl_hpo_results[name]['model']
        def dl_predict_covid(model, X_df, scaler_X=scaler_X, scaler_y=scaler_y):
            X_sc = scaler_X.transform(X_df)
            y_dummy = np.zeros(len(X_sc))
            ds = TimeSeriesDataset(X_sc, y_dummy, WINDOW_SIZE)
            loader = DataLoader(ds, batch_size=256, shuffle=False)
            return predict_dl(model, loader, scaler_y)

        covid_models[f'{name} (Deep Learning)'] = {
            'model': dl_model_covid,
            'family': 'Deep Learning',
            'predict_fn': dl_predict_covid,
            'needs_df': False,
            'is_dl': True,
        }
        break

print(f"Modelos para evaluación COVID: {list(covid_models.keys())}")

# --- Generar predicciones por período ---
covid_results = {}

for pid, info in COVID_PERIODS.items():
    if not info.get('available', False):
        continue

    mask = (df_covid_clean.index >= info['start']) & (df_covid_clean.index <= info['end'])
    df_period = df_covid_clean.loc[mask]
    X_period = df_period[available_feats]
    y_period = df_period[TARGET]

    period_results = {'y_true': y_period}

    for model_name, minfo in covid_models.items():
        try:
            if model_name.startswith('MoE'):
                # MoE necesita el DataFrame completo
                preds = minfo['predict_fn'](minfo['model'], df_period)
            elif minfo.get('is_dl', False):
                # DL necesita ventana temporal
                preds_raw = minfo['predict_fn'](minfo['model'], X_period)
                # Ajustar longitud (DL pierde WINDOW_SIZE muestras al inicio)
                if len(preds_raw) < len(y_period):
                    # Pad con NaN al inicio
                    preds = np.full(len(y_period), np.nan)
                    preds[-len(preds_raw):] = preds_raw
                else:
                    preds = preds_raw[:len(y_period)]
            else:
                preds = minfo['predict_fn'](minfo['model'], X_period)

            # Métricas (excluyendo NaN para DL)
            valid_mask = ~np.isnan(preds) if isinstance(preds, np.ndarray) else np.ones(len(preds), dtype=bool)
            if valid_mask.sum() > 0:
                metrics = compute_metrics(y_period.values[valid_mask], preds[valid_mask])
            else:
                metrics = {'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan, 'R2': np.nan}

            period_results[model_name] = {'preds': preds, 'metrics': metrics}
        except Exception as e:
            print(f"  ⚠️ {model_name} falló en {pid}: {e}")
            period_results[model_name] = {
                'preds': np.full(len(y_period), np.nan),
                'metrics': {'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan, 'R2': np.nan}
            }

    covid_results[pid] = period_results

print("\n✓ Predicciones generadas para todos los períodos")

In [ ]:
# ============================================================
# 10.5.3  TABLA COMPARATIVA POR PERÍODO
# ============================================================
print("\n" + "=" * 110)
print("  RESULTADOS — STRESS TEST COVID-19 (datos nunca vistos por los modelos)")
print("=" * 110)

covid_summary_rows = []

for pid, info in COVID_PERIODS.items():
    if pid not in covid_results:
        continue
    results = covid_results[pid]
    print(f"\n  ▸ {pid}: {info['description']}")
    print(f"    Período: {info['start']} → {info['end']} | {info.get('n_hours', 0):,}h")
    print(f"    Impacto esperado: {info['expected']}")
    print(f"    {'Modelo':30s} | {'MAPE (%)':>10s} | {'RMSE':>12s} | {'MAE':>12s} | {'R²':>8s}")
    print(f"    {'-'*30}-+-{'-'*10}-+-{'-'*12}-+-{'-'*12}-+-{'-'*8}")

    for model_name in covid_models.keys():
        if model_name in results:
            m = results[model_name]['metrics']
            print(f"    {model_name:30s} | {m['MAPE']:10.4f} | {m['RMSE']:12.2f} | "
                  f"{m['MAE']:12.2f} | {m['R2']:8.4f}")
            covid_summary_rows.append({
                'Período': pid,
                'Descripción': info['description'],
                'Modelo': model_name,
                'Familia': covid_models[model_name]['family'],
                'MAPE (%)': m['MAPE'],
                'RMSE': m['RMSE'],
                'MAE': m['MAE'],
                'R²': m['R2'],
            })

covid_summary_df = pd.DataFrame(covid_summary_rows)
covid_summary_df.to_csv(BENCH_DIR / 'covid_stress_test_results.csv', index=False)
print(f"\n✓ Resultados guardados en {BENCH_DIR / 'covid_stress_test_results.csv'}")

#### 9.5.3 Visualización: Series Temporales por Período Crítico

In [ ]:
# ============================================================
# 9.5.4  SERIES TEMPORALES — PREDICCIONES vs REAL
# ============================================================
available_periods = [pid for pid, info in COVID_PERIODS.items() if info.get('available', False)]
n_periods = len(available_periods)

fig, axes = plt.subplots(n_periods, 1, figsize=(18, 5 * n_periods), sharex=False)
if n_periods == 1:
    axes = [axes]

model_colors = {}
for i, (name, minfo) in enumerate(covid_models.items()):
    model_colors[name] = family_colors.get(minfo['family'], f'C{i}')

for ax, pid in zip(axes, available_periods):
    info = COVID_PERIODS[pid]
    results = covid_results[pid]
    y_true = results['y_true']

    # Plot real
    ax.plot(y_true.index, y_true.values, color='black', linewidth=1.5,
            label='Real', alpha=0.9, zorder=10)

    # Plot predicciones de cada modelo
    for model_name in covid_models.keys():
        if model_name in results:
            preds = results[model_name]['preds']
            mape = results[model_name]['metrics']['MAPE']
            color = model_colors[model_name]
            label_str = f"{model_name} (MAPE: {mape:.2f}%)"
            ax.plot(y_true.index, preds, color=color, linewidth=1.0,
                    alpha=0.75, label=label_str, linestyle='--')

    # Formato
    ax.set_title(f"{pid}: {info['description']}", fontweight='bold', fontsize=12,
                 color=info['color'])
    ax.set_ylabel('Demanda (kWh)')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

    # Sombrear el período
    ax.axvspan(pd.Timestamp(info['start']), pd.Timestamp(info['end']),
               alpha=0.05, color=info['color'])

fig.suptitle('Stress Test COVID-19: Predicciones vs. Demanda Real\n'
             '(Datos 2020-2021 nunca vistos durante entrenamiento)',
             fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(BENCH_DIR / 'fig_covid_timeseries.png', bbox_inches='tight')
plt.show()
print("✓ Figura guardada: fig_covid_timeseries.png")

#### 9.5.4 Análisis de Degradación por Período

In [ ]:
# ============================================================
# 9.5.5  HEATMAP DE DEGRADACIÓN — MAPE POR PERÍODO × MODELO
# ============================================================
if len(covid_summary_df) > 0:
    # Pivot: filas=períodos, columnas=modelos, valores=MAPE
    pivot_mape = covid_summary_df.pivot_table(
        index='Período', columns='Modelo', values='MAPE (%)', aggfunc='first'
    )

    # Ordenar períodos cronológicamente
    period_order = [p for p in COVID_PERIODS.keys() if p in pivot_mape.index]
    pivot_mape = pivot_mape.reindex(period_order)

    # Labels descriptivos
    period_labels = [f"{pid}\n{COVID_PERIODS[pid]['description'][:35]}"
                     for pid in period_order]

    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot_mape, annot=True, fmt='.2f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'MAPE (%)'},
                yticklabels=period_labels)
    ax.set_title('Degradación de Modelos en Períodos COVID-19\n'
                 '(valores más altos = peor desempeño OOD)',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Modelo')
    ax.set_ylabel('')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_covid_degradation_heatmap.png', bbox_inches='tight')
    plt.show()

    # --- MAPE relativo: cuánto se degrada vs test set normal ---
    print("\n" + "=" * 90)
    print("  DEGRADACIÓN RELATIVA — MAPE COVID vs. MAPE Test Normal")
    print("=" * 90)
    print(f"  {'Modelo':30s} | {'MAPE Test':>10s} | {'MAPE COVID':>11s} | {'Degradación':>12s}")
    print(f"  {'-'*30}-+-{'-'*10}-+-{'-'*11}-+-{'-'*12}")

    for model_name in covid_models.keys():
        # MAPE en test normal
        for fname, r in final_results.items():
            if model_name.split(' (')[0] in fname:
                mape_normal = r['test_metrics']['MAPE']
                break
        else:
            mape_normal = np.nan

        # MAPE promedio en COVID
        model_covid = covid_summary_df[covid_summary_df['Modelo'] == model_name]
        mape_covid_avg = model_covid['MAPE (%)'].mean()

        if not np.isnan(mape_normal) and mape_normal > 0:
            degradation = ((mape_covid_avg - mape_normal) / mape_normal) * 100
            print(f"  {model_name:30s} | {mape_normal:10.4f} | {mape_covid_avg:11.4f} | "
                  f"{degradation:+11.1f}%")
        else:
            print(f"  {model_name:30s} | {'N/A':>10s} | {mape_covid_avg:11.4f} | {'N/A':>12s}")

In [ ]:
# ============================================================
# 9.5.6  BOXPLOT — DISTRIBUCIÓN DE ERROR POR PERÍODO
# ============================================================
fig, axes = plt.subplots(1, len(covid_models), figsize=(6 * len(covid_models), 7),
                         sharey=True)
if len(covid_models) == 1:
    axes = [axes]

for ax, (model_name, minfo) in zip(axes, covid_models.items()):
    errors_by_period = []
    period_names = []

    for pid in period_order:
        if pid in covid_results and model_name in covid_results[pid]:
            preds = covid_results[pid][model_name]['preds']
            y_true = covid_results[pid]['y_true'].values
            valid = ~np.isnan(preds)
            if valid.sum() > 0:
                abs_pct_error = np.abs((y_true[valid] - preds[valid]) / y_true[valid]) * 100
                errors_by_period.append(abs_pct_error)
                short_label = COVID_PERIODS[pid]['label'].replace('\n', ' ')
                period_names.append(short_label)

    if errors_by_period:
        bp = ax.boxplot(errors_by_period, labels=period_names, patch_artist=True,
                        showfliers=False, medianprops=dict(color='black', linewidth=2))
        for i, (patch, pid) in enumerate(zip(bp['boxes'], period_order)):
            patch.set_facecolor(COVID_PERIODS[pid]['color'])
            patch.set_alpha(0.6)

    ax.set_title(f'{model_name}', fontweight='bold', fontsize=11)
    ax.set_ylabel('|Error Porcentual| (%)' if ax == axes[0] else '')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Distribución del Error por Período COVID — Comparación entre Modelos',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(BENCH_DIR / 'fig_covid_error_boxplots.png', bbox_inches='tight')
plt.show()

#### 9.5.5 Análisis Detallado: Semana Santa bajo Confinamiento (Peor Caso)

In [ ]:
# ============================================================
# 9.5.7  ZOOM: SEMANA SANTA 2020 (P3) — CASO EXTREMO
# ============================================================
P3_ID = 'P3_SemanaSanta'

if P3_ID in covid_results:
    results_p3 = covid_results[P3_ID]
    y_true_p3 = results_p3['y_true']

    fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})

    # Panel superior: series temporales
    ax = axes[0]
    ax.plot(y_true_p3.index, y_true_p3.values, color='black', linewidth=2,
            label='Demanda Real', zorder=10)

    for model_name in covid_models.keys():
        if model_name in results_p3:
            preds = results_p3[model_name]['preds']
            mape = results_p3[model_name]['metrics']['MAPE']
            color = model_colors[model_name]
            ax.plot(y_true_p3.index, preds, color=color, linewidth=1.2,
                    alpha=0.8, label=f"{model_name} (MAPE: {mape:.2f}%)", linestyle='--')

    ax.set_ylabel('Demanda (kWh)', fontsize=12)
    ax.set_title('Semana Santa 2020 bajo Confinamiento Total — Caso Extremo OOD',
                 fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)

    # Anotar días
    for day_offset in range(7):
        day = pd.Timestamp('2020-04-06') + pd.Timedelta(days=day_offset)
        day_names = ['Lun Santo', 'Mar Santo', 'Mié Santo', 'Jue Santo',
                     'Vie Santo', 'Sáb Gloria', 'Dom Resurrección']
        if day in y_true_p3.index or day <= y_true_p3.index.max():
            ax.axvline(x=day, color='gray', alpha=0.3, linestyle=':')
            ax.text(day, ax.get_ylim()[0], day_names[day_offset],
                    rotation=45, fontsize=7, ha='left', va='bottom')

    # Panel inferior: error porcentual horario
    ax = axes[1]
    for model_name in covid_models.keys():
        if model_name in results_p3:
            preds = results_p3[model_name]['preds']
            valid = ~np.isnan(preds)
            error_pct = ((preds[valid] - y_true_p3.values[valid]) /
                         y_true_p3.values[valid]) * 100
            idx_valid = y_true_p3.index[valid]
            ax.plot(idx_valid, error_pct, color=model_colors[model_name],
                    alpha=0.7, linewidth=1)

    ax.axhline(y=0, color='black', linewidth=1)
    ax.axhline(y=5, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(y=-5, color='red', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.set_ylabel('Error (%)')
    ax.set_xlabel('Hora')
    ax.fill_between(ax.get_xlim(), -5, 5, alpha=0.05, color='green')

    plt.tight_layout()
    plt.savefig(BENCH_DIR / 'fig_covid_semana_santa_zoom.png', bbox_inches='tight')
    plt.show()

    # Tabla horaria resumen
    print("\nDesempeño horario Semana Santa 2020:")
    print(f"  Demanda real — Media: {y_true_p3.mean():,.0f} | Min: {y_true_p3.min():,.0f} | "
          f"Max: {y_true_p3.max():,.0f}")
    for model_name in covid_models.keys():
        if model_name in results_p3:
            m = results_p3[model_name]['metrics']
            print(f"  {model_name:30s} — MAPE: {m['MAPE']:.2f}% | "
                  f"RMSE: {m['RMSE']:,.0f} | R²: {m['R2']:.4f}")
else:
    print("⚠️ Período P3 no disponible en los datos")

#### 9.5.6 Conclusiones del Stress Test COVID

In [ ]:
# ============================================================
# 9.5.8  RESUMEN Y CONCLUSIONES DEL STRESS TEST
# ============================================================
print("\n" + "=" * 100)
print("  CONCLUSIONES — STRESS TEST COVID-19 (Out-of-Distribution)")
print("=" * 100)

if len(covid_summary_df) > 0:
    # 1. Mejor modelo por período
    print("\n  1. MEJOR MODELO POR PERÍODO:")
    for pid in period_order:
        sub = covid_summary_df[covid_summary_df['Período'] == pid]
        if len(sub) > 0:
            best = sub.loc[sub['MAPE (%)'].idxmin()]
            print(f"     {pid:20s} → {best['Modelo']:30s} (MAPE: {best['MAPE (%)']:.2f}%)")

    # 2. Modelo más robusto (menor MAPE promedio across períodos)
    print("\n  2. ROBUSTEZ GLOBAL (MAPE promedio across períodos COVID):")
    avg_by_model = covid_summary_df.groupby('Modelo')['MAPE (%)'].agg(['mean', 'std'])
    avg_by_model = avg_by_model.sort_values('mean')
    for name, row in avg_by_model.iterrows():
        print(f"     {name:30s} — MAPE medio: {row['mean']:.2f}% ± {row['std']:.2f}%")

    # 3. Período más difícil
    print("\n  3. PERÍODO MÁS DIFÍCIL (mayor MAPE promedio across modelos):")
    avg_by_period = covid_summary_df.groupby('Período')['MAPE (%)'].mean().sort_values(ascending=False)
    for pid, mape_avg in avg_by_period.items():
        print(f"     {pid:20s} — MAPE medio: {mape_avg:.2f}%  "
              f"({COVID_PERIODS[pid]['description'][:50]})")

    # 4. Ventaja del MoE
    print("\n  4. ANÁLISIS DE LA ARQUITECTURA MoE BAJO ESTRÉS:")
    moe_rows = covid_summary_df[covid_summary_df['Familia'] == 'MoE']
    if len(moe_rows) > 0:
        moe_avg = moe_rows['MAPE (%)'].mean()
        others_avg = covid_summary_df[covid_summary_df['Familia'] != 'MoE']['MAPE (%)'].mean()
        advantage = others_avg - moe_avg
        print(f"     MAPE medio MoE:     {moe_avg:.2f}%")
        print(f"     MAPE medio otros:   {others_avg:.2f}%")
        if advantage > 0:
            print(f"     Ventaja MoE:        {advantage:.2f} puntos porcentuales")
            print(f"     El gating adaptativo permite degradación más suave en OOD,")
            print(f"     ya que el soft gating distribuye la predicción entre expertos")
            print(f"     cuando la confianza del clasificador es baja.")
        else:
            print(f"     Diferencia:         {advantage:.2f} pp (modelos convencionales superiores)")
            print(f"     En datos OOD extremos, modelos más simples pueden ser más")
            print(f"     robustos al no depender de un clasificador de clusters.")

    # Guardar resumen
    covid_conclusions = {
        'best_per_period': {pid: covid_summary_df[covid_summary_df['Período'] == pid]
                            .nsmallest(1, 'MAPE (%)')['Modelo'].iloc[0]
                            for pid in period_order
                            if pid in covid_summary_df['Período'].values},
        'most_robust': avg_by_model.index[0],
        'hardest_period': avg_by_period.index[0],
        'avg_mape_by_model': avg_by_model['mean'].to_dict(),
    }
    with open(BENCH_DIR / 'covid_stress_conclusions.json', 'w') as f:
        json.dump(covid_conclusions, f, indent=2, default=str)

print("\n" + "=" * 100)

---
## 10. Análisis predictivo para Enero

### Funciones

In [ ]:
import holidays # Import the holidays library

print("Generando predicciones para enero de 2026...")

# 1. Definir el rango de fechas para la predicción (Enero 2026)
start_date_pred = pd.Timestamp('2026-01-01 00:00:00')
end_date_pred = pd.Timestamp('2026-01-31 23:00:00')
future_index = pd.date_range(start=start_date_pred, end=end_date_pred, freq='H')

# Obtener los días festivos de Colombia para un rango relevante
# Incluye años anteriores para 'days_from_prev_holiday' y futuros para 'days_to_next_holiday'
co_holidays = holidays.CO(years=range(2020, 2027))

# Crear un DataFrame para almacenar todas las predicciones de todos los modelos
df_pred_all_models = pd.DataFrame(index=future_index)

In [ ]:
def generate_features_for_timestamp(ts, historical_df, future_df, feature_cols, target_col, co_holidays):
    """Genera las características para un timestamp dado.
    historical_df: DataFrame con datos históricos reales (hasta el final del entrenamiento)
    future_df: DataFrame con las predicciones ya generadas para el futuro (para lags)
    """
    features = {}

    # 1. Características cíclicas de tiempo
    features['hour_sin'] = np.sin(2 * np.pi * ts.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * ts.hour / 24)
    features['dayofweek_sin'] = np.sin(2 * np.pi * ts.dayofweek / 7)
    features['dayofweek_cos'] = np.cos(2 * np.pi * ts.dayofweek / 7)
    features['month_sin'] = np.sin(2 * np.pi * ts.month / 12)
    features['month_cos'] = np.cos(2 * np.pi * ts.month / 12)

    # 2. Características de días festivos
    features['is_holiday'] = 1 if ts.date() in co_holidays else 0

    features['is_bridge'] = 0
    if ts.weekday() == 0 and (ts.date() - timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if ts.weekday() == 4 and (ts.date() + timedelta(days=3)) in co_holidays:
        features['is_bridge'] = 1
    if (ts.date() - timedelta(days=1)) in co_holidays and (ts.date() + timedelta(days=1)) in co_holidays:
        features['is_bridge'] = 1

    features['days_to_next_holiday'] = 0
    features['days_from_prev_holiday'] = 0

    if ts.date() not in co_holidays:
        next_holiday_date = None
        for day_offset in range(1, 366):
            future_date = ts.date() + timedelta(days=day_offset)
            if future_date in co_holidays:
                next_holiday_date = future_date
                break
        if next_holiday_date:
            features['days_to_next_holiday'] = (next_holiday_date - ts.date()).days
        else:
            features['days_to_next_holiday'] = 365

        prev_holiday_date = None
        for day_offset in range(1, 366):
            past_date = ts.date() - timedelta(days=day_offset)
            if past_date in co_holidays:
                prev_holiday_date = past_date
                break
        if prev_holiday_date:
            features['days_from_prev_holiday'] = (ts.date() - prev_holiday_date).days
        else:
            features['days_from_prev_holiday'] = 365

    # 3. Características de rezago (lags)
    # Asegurarse de que los índices son datetime para concatenar y buscar
    if not isinstance(historical_df.index, pd.DatetimeIndex):
        historical_df.index = pd.to_datetime(historical_df.index)
    if not isinstance(future_df.index, pd.DatetimeIndex):
        future_df.index = pd.to_datetime(future_df.index)

    # Combinar datos históricos y futuros para la búsqueda de lags
    # Se usa historical_df[target_col] para asegurarse de que solo se toman los valores reales de demanda
    # future_df[target_col].dropna() es crucial para que solo se incluyan las predicciones ya hechas
    combined_data = pd.concat([historical_df[target_col], future_df[target_col].dropna()]).sort_index()

    lag_cols_nums = sorted([int(c.split('_')[-1]) for c in feature_cols if c.startswith('demand_lag_')])
    for lag in lag_cols_nums:
        lag_ts = ts - timedelta(hours=lag)
        if lag_ts in combined_data.index:
            features[f'demand_lag_{lag}'] = combined_data.loc[lag_ts]
        else:
            # Si el lag no está disponible (ej. ts es muy temprano en el historial), usar NaN.
            # Los modelos deben ser robustos a NaNs o se debería pre-procesar.
            # Para este caso, df_clean cubre el rango necesario.
            features[f'demand_lag_{lag}'] = np.nan

    # Solo incluir características que están en FEATURE_COLS
    return {k: v for k, v in features.items() if k in feature_cols}

In [ ]:
from typing import List, Tuple, Optional, Dict, Any
import requests

# ============================================================
# Funciones de ingestión — API XM (servapibi.xm.com.co)
# Correcciones alineadas con el cliente JS de referencia (ReadDB)
# ============================================================
API_URL = "https://servapibi.xm.com.co/hourly"
DEMAND_VARIABLE = "DemaReal"
MAX_RETRIES = 3
INITIAL_BACKOFF = 2  # segundos

# Rango histórico válido de demanda horaria del SIN (kWh): ~4.000-12.000 MWh
DEMAND_MIN_KWH = 4_000_000
DEMAND_MAX_KWH = 12_000_000


def generate_date_windows(
    start_date: str, end_date: str
) -> List[Tuple[str, str]]:
    """
    Genera ventanas de fechas para iterar sobre la API de XM.

    *** FIX CRÍTICO ***
    El cliente JS de referencia genera ventanas de UN SOLO DÍA para datos
    horarios (HourlyEntities). La versión anterior usaba ventanas de 30 días,
    lo cual probablemente excede el límite interno de la API (30 × 24 = 720
    registros por request), causando respuestas vacías o truncadas.

    Ahora cada ventana cubre exactamente 1 día → 24 registros por request,
    consistente con el comportamiento exitoso del cliente JS.

    Parameters
    ----------
    start_date : str  — Fecha inicio 'YYYY-MM-DD'.
    end_date : str    — Fecha fin 'YYYY-MM-DD'.

    Returns
    -------
    List[Tuple[str, str]]
        Lista de tuplas (inicio, fin) donde inicio == fin (un día por ventana).
    """
    windows = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        # Ventana de 1 solo día: start == end (alineado con JS)
        date_str = current.strftime("%Y-%m-%d")
        windows.append((date_str, date_str))
        current += timedelta(days=1)

    return windows


def _parse_hourly_items(items: list) -> list:
    """
    Parsea la respuesta de la API de XM, soportando AMBOS formatos
    posibles de HourlyEntities:

    Formato A (lista de {Hour, Value}):
        "HourlyEntities": [
            {"Hour": 1, "Value": 5432.1},
            {"Hour": 2, "Value": 5210.3}, ...
        ]

    Formato B (lista de entidades con Values dict):
        "HourlyEntities": [
            {
                "Id": "Sistema",
                "Values": {"Hour01": 5432.1, "Hour02": 5210.3, ...}
            }
        ]

    Este parsing defensivo evita fallos silenciosos si la estructura
    de la respuesta no coincide con lo esperado.
    """
    records = []

    for item in items:
        date_str = item.get("Date", "")
        if not date_str:
            continue

        base_date = pd.to_datetime(date_str).normalize()
        hourly_entities = item.get("HourlyEntities", [])

        for entity in hourly_entities:
            # --- Formato A: cada entity tiene Hour y Value directamente ---
            if "Hour" in entity and "Value" in entity:
                hour = entity["Hour"]  # Horas 1-24 en la API
                value = entity.get("Value")
                if value is not None:
                    dt = base_date + pd.Timedelta(hours=int(hour) - 1)
                    records.append({"datetime": dt, "demand_kwh": float(value)})

            # --- Formato B: entity tiene Id + Values dict ---
            elif "Values" in entity:
                values_dict = entity["Values"]
                for key, value in values_dict.items():
                    # Claves tipo "Hour01", "Hour02", ..., "Hour24"
                    if key.startswith("Hour") and value is not None:
                        try:
                            hour = int(key.replace("Hour", ""))
                            dt = base_date + pd.Timedelta(hours=hour - 1)
                            records.append({"datetime": dt, "demand_kwh": float(value)})
                        except (ValueError, TypeError):
                            continue

            # --- Formato desconocido: logear para debug ---
            else:
                print(f"  [WARN] Estructura inesperada en HourlyEntities: {list(entity.keys())}")

    return records


def fetch_xm_demand(
    start_date: str,
    end_date: str,
    max_retries: int = MAX_RETRIES,
    initial_backoff: float = INITIAL_BACKOFF,
) -> Optional[pd.DataFrame]:
    """
    Descarga datos horarios de demanda real del SIN desde la API de XM
    para una ventana de fechas (idealmente 1 día).

    *** FIXES APLICADOS ***
    1. Se agrega el campo "Filter": [] al payload (presente en el cliente JS,
       ausente en la versión anterior). Algunas APIs validan la presencia de
       todos los campos del schema aunque estén vacíos.
    2. Parsing defensivo con soporte para múltiples formatos de respuesta.

    Parameters
    ----------
    start_date : str  — Fecha inicio 'YYYY-MM-DD'.
    end_date : str    — Fecha fin 'YYYY-MM-DD'.
    max_retries : int — Número máximo de reintentos.
    initial_backoff : float — Backoff inicial en segundos.

    Returns
    -------
    pd.DataFrame or None
    """
    # FIX: agregar "Filter": [] al payload, alineado con el cliente JS
    payload = {
        "MetricId": DEMAND_VARIABLE,
        "StartDate": start_date,
        "EndDate": end_date,
        "Entity": "Sistema",
        "Filter": [],  # ← Campo presente en el JS, faltaba en Python
    }
    headers = {"Content-Type": "application/json"}

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                API_URL, json=payload, headers=headers, timeout=60
            )
            response.raise_for_status()
            data = response.json()

            # Parsing defensivo que soporta ambos formatos de respuesta
            items = data.get("Items", [])
            records = _parse_hourly_items(items)

            if records:
                return pd.DataFrame(records)
            else:
                print(f"  [WARN] Sin registros para {start_date} -> {end_date}")
                return None

        except requests.exceptions.RequestException as e:
            wait = initial_backoff * (2 ** (attempt - 1))
            print(
                f"  [RETRY {attempt}/{max_retries}] Error: {e}. "
                f"Reintentando en {wait:.0f}s..."
            )
            time.sleep(wait)

    print(f"  [ERROR] Fallaron todos los intentos para {start_date} -> {end_date}")
    return None


def download_demand_data(
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """
    Descarga datos de demanda horaria del SIN para un rango de fechas
    arbitrario, iterando día a día (alineado con el cliente JS).

    Nota sobre eficiencia: con ventanas de 1 día, un año requiere ~365
    requests. Con el sleep de 0.3s por request, eso toma ~2 min.
    Es el tradeoff correcto: la API responde consistentemente vs. ventanas
    grandes que devuelven vacío.

    Parameters
    ----------
    start_date : str  — Fecha inicio 'YYYY-MM-DD'.
    end_date : str    — Fecha fin 'YYYY-MM-DD'.

    Returns
    -------
    pd.DataFrame
        DataFrame consolidado ['datetime', 'demand_kwh'], ordenado, sin duplicados.
    """
    windows = generate_date_windows(start_date, end_date)
    print(f"Total de ventanas (días) a consultar: {len(windows)}")
    print(f"Período: {start_date} -> {end_date}")
    print("=" * 60)

    all_dfs = []
    failed_days = []

    for i, (ws, we) in enumerate(windows, 1):
        #print(f"[{i:3d}/{len(windows)}] Descargando {ws} ...", end=" ")
        df = fetch_xm_demand(ws, we)
        if df is not None:
            all_dfs.append(df)
            #print(f"OK ({len(df)} registros)")
        else:
            failed_days.append(ws)
            print("SIN DATOS")

        # Pausa entre requests para no saturar la API
        # (el JS usa Promise.all sin throttling, pero ser conservador es mejor)
        time.sleep(0.3)

    if not all_dfs:
        raise RuntimeError("No se obtuvieron datos de la API.")

    # Consolidar
    df_full = pd.concat(all_dfs, ignore_index=True)

    # Eliminar duplicados por timestamp
    n_before = len(df_full)
    df_full = df_full.drop_duplicates(subset="datetime", keep="first")
    n_dupes = n_before - len(df_full)
    if n_dupes > 0:
        print(f"\n[INFO] Se eliminaron {n_dupes} timestamps duplicados.")

    # Ordenar
    df_full = df_full.sort_values("datetime").reset_index(drop=True)

    print(f"\n{'=' * 60}")
    print(f"Descarga completada: {len(df_full):,} registros")
    print(f"Rango: {df_full['datetime'].min()} -> {df_full['datetime'].max()}")

    if failed_days:
        print(f"[WARN] Días sin datos ({len(failed_days)}): {failed_days[:10]}...")

    return df_full

### Predicciones MoE

In [ ]:
# Crear un DataFrame para almacenar las características futuras y predicciones
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Generar características para el timestamp actual
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays)

    # Convertir a DataFrame de una fila para la predicción
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # Realizar la predicción MoE
    pred = moe_model.predict(X_current)[0]
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Pronosticado {i + 1}/{len(future_index)} horas para Enero 2026. Última predicción: {ts} -> {pred:,.0f} kWh")

print("\nPredicción de Enero 2026 completada Para el MoE.")

df_pred_all_models['pred_moe'] = df_pred_features[TARGET]

# Visualizar la predicción
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Predicción de Demanda (kWh)', color='blue')
plt.title('Pronóstico de Demanda Eléctrica para Enero 2026 (MoE)')
plt.xlabel('Fecha')
plt.ylabel('Demanda (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Predicciones Lasso

In [ ]:
model_lasso = list(hpo_results.items())[0][1]['model']

In [ ]:
# Crear un DataFrame para almacenar las características futuras y predicciones
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Generar características para el timestamp actual
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays)

    # Convertir a DataFrame de una fila para la predicción
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # Realizar la predicción Lasso
    pred = model_lasso.predict(X_current)[0]
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Pronosticado {i + 1}/{len(future_index)} horas para Enero 2026. Última predicción: {ts} -> {pred:,.0f} kWh")

print("\nPredicción de Enero 2026 completada Para el Lasso.")

df_pred_all_models['pred_lasso'] = df_pred_features[TARGET]

# Visualizar la predicción
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Predicción de Demanda (kWh)', color='blue')
plt.title('Pronóstico de Demanda Eléctrica para Enero 2026 (Lasso)')
plt.xlabel('Fecha')
plt.ylabel('Demanda (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Predicciones Adaboost

In [ ]:
model_adaboost = list(hpo_results.items())[1][1]['model']

In [ ]:
# Crear un DataFrame para almacenar las características futuras y predicciones
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

for i, ts in enumerate(future_index):
    # Generar características para el timestamp actual
    current_features_dict = generate_features_for_timestamp(ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays)

    # Convertir a DataFrame de una fila para la predicción
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # Realizar la predicción Adaboost
    pred = model_adaboost.predict(X_current)[0]
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Pronosticado {i + 1}/{len(future_index)} horas para Enero 2026. Última predicción: {ts} -> {pred:,.0f} kWh")

print("\nPredicción de Enero 2026 completada Para el Adaboost.")

df_pred_all_models['pred_adaboost'] = df_pred_features[TARGET]

# Visualizar la predicción
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Predicción de Demanda (kWh)', color='blue')
plt.title('Pronóstico de Demanda Eléctrica para Enero 2026 (Adaboost)')
plt.xlabel('Fecha')
plt.ylabel('Demanda (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Predicciones Transformers

In [ ]:
model_transformers = list(dl_hpo_results.items())[0][1]['model']

In [ ]:
df_pred_features = pd.DataFrame(index=future_index)

if not isinstance(df_clean.index, pd.DatetimeIndex):
    df_clean.index = pd.to_datetime(df_clean.index)

model_transformers.eval()

# Crear un DataFrame para almacenar las características futuras y predicciones
df_pred_features = pd.DataFrame(index=future_index, columns=calendar_lags + [TARGET])

for i, ts in enumerate(future_index):
    # Generar características para el timestamp actual
    current_features_dict = generate_features_for_timestamp(
        ts, df_clean, df_pred_features, calendar_lags, TARGET, co_holidays
    )

    # Convertir a DataFrame de una fila
    X_current = pd.DataFrame([current_features_dict], index=[ts], columns=calendar_lags)

    # Escalar la fila actual con el scaler de entrada
    X_current_sc = scaler_X.transform(X_current)

    # ---------------------------------------------------------
    # Construir la secuencia de tamaño WINDOW_SIZE
    # usando histórico real + predicciones ya generadas
    # ---------------------------------------------------------
    if i == 0:
        hist_input = df_clean[calendar_lags].iloc[-(WINDOW_SIZE-1):].copy()
        X_seq = pd.concat([hist_input, X_current], axis=0)
    else:
        prev_rows = df_pred_features[calendar_lags].iloc[:i].copy()

        hist_needed = WINDOW_SIZE - len(prev_rows) - 1

        if hist_needed > 0:
            hist_input = df_clean[calendar_lags].iloc[-hist_needed:].copy()
            X_seq = pd.concat([hist_input, prev_rows, X_current], axis=0)
        else:
            X_seq = pd.concat([prev_rows.iloc[-(WINDOW_SIZE-1):], X_current], axis=0)

    # Asegurar orden exacto de columnas
    X_seq = X_seq[calendar_lags]

    # Escalar secuencia completa
    X_seq_sc = scaler_X.transform(X_seq)

    # Convertir a tensor con forma (1, WINDOW_SIZE, n_features)
    X_tensor = torch.tensor(X_seq_sc, dtype=torch.float32).unsqueeze(0).to(device)

    # Predicción DL
    with torch.no_grad():
        pred_sc = model_transformers(X_tensor).cpu().numpy().reshape(-1, 1)

    pred = scaler_y.inverse_transform(pred_sc).flatten()[0]

    # Guardar predicción
    df_pred_features.loc[ts, TARGET] = pred

    if (i + 1) % 100 == 0 or i == len(future_index) - 1:
        print(f"  Pronosticado {i + 1}/{len(future_index)} horas para Enero 2026. Última predicción: {ts} -> {pred:,.0f} kWh")


print(f"\nPredicción de Enero 2026 completada para Transformers.")

df_pred_all_models['pred_transformers'] = df_pred_features[TARGET]

# Visualizar la predicción
plt.figure(figsize=(15, 6))
plt.plot(df_pred_features.index, df_pred_features[TARGET], label='Predicción de Demanda (kWh)', color='blue')
plt.title('Pronóstico de Demanda Eléctrica para Enero 2026 (Transformers)')
plt.xlabel('Fecha')
plt.ylabel('Demanda (kWh)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Descarga de datos XM

In [ ]:
# Descargar datos
START_DATE = "2026-01-01"
END_DATE = "2026-01-31"

df_demand_real_january = download_demand_data(START_DATE, END_DATE)

In [ ]:
# Asegurarse de que el índice de df_demand_real_january sea datetime y esté alineado
df_demand_real_january = df_demand_real_january.set_index('datetime').sort_index()

In [ ]:
df_demand_real_january

### Comparación

In [ ]:
df_pred_all_models

#### Graficos

In [ ]:
plt.figure(figsize=(15, 8))

# Plot de las predicciones
for col in df_pred_all_models.columns:
    plt.plot(df_pred_all_models.index, df_pred_all_models[col], label=f"Predicción {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

# Plot de la demanda real si está disponible
if 'df_demand_real_january' in locals() and not df_demand_real_january.empty:
    plt.plot(df_demand_real_january.index, df_demand_real_january['demand_kwh'], label='Demanda Real (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Pronóstico de Demanda Eléctrica para Enero 2026', fontsize=16)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Demanda (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig('demand_forecast_january_2026.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))

for col in df_pred_all_models.columns:
    plt.plot(df_pred_all_models.index, df_pred_all_models[col], label=f"Prediction {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

if 'df_demand_real_january' in locals() and not df_demand_real_january.empty:
    plt.plot(df_demand_real_january.index, df_demand_real_january['demand_kwh'], label='Actual Demand (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Electricity Demand Forecast for January 2026', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig('demand_forecast_january_2026_english.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

# Define the first week
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-07 23:00:00')

# Filter predictions for the first week
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Filter real demand for the first week
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Plot de las predicciones
for col in df_pred_week1.columns:
    plt.plot(df_pred_week1.index, df_pred_week1[col], label=f"Predicción {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

# Plot de la demanda real si está disponible
if not df_demand_real_week1.empty:
    plt.plot(df_demand_real_week1.index, df_demand_real_week1['demand_kwh'], label='Demanda Real (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Pronóstico de Demanda Eléctrica para la Primera Semana de Enero 2026 - Comparación de Modelos', fontsize=16)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Demanda (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig('demand_forecast_january_2026_one_week.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-07 23:00:00')

df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

for col in df_pred_week1.columns:
    plt.plot(df_pred_week1.index, df_pred_week1[col], label=f"Prediction {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

if not df_demand_real_week1.empty:
    plt.plot(df_demand_real_week1.index, df_demand_real_week1['demand_kwh'], label='Actual Demand (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Electricity Demand Forecast for the First Week of January 2026 — Model Comparison', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig('demand_forecast_january_2026_one_week_english.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

# Define the first week
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-14 23:00:00')

# Filter predictions for the first week
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Filter real demand for the first week
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Plot de las predicciones
for col in df_pred_week1.columns:
    plt.plot(df_pred_week1.index, df_pred_week1[col], label=f"Predicción {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

# Plot de la demanda real si está disponible
if not df_demand_real_week1.empty:
    plt.plot(df_demand_real_week1.index, df_demand_real_week1['demand_kwh'], label='Demanda Real (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Pronóstico de Demanda Eléctrica para las Primeras 2 Semanas de Enero 2026 - Comparación de Modelos', fontsize=16)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Demanda (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig('demand_forecast_january_2026_two_week.png')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-14 23:00:00')

df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

for col in df_pred_week1.columns:
    plt.plot(df_pred_week1.index, df_pred_week1[col], label=f"Prediction {col.replace('pred_', '', 1).capitalize().replace('Moe', 'MoE')}", alpha=0.7)

if not df_demand_real_week1.empty:
    plt.plot(df_demand_real_week1.index, df_demand_real_week1['demand_kwh'], label='Actual Demand (XM)', color='black', linewidth=2, alpha=0.8)

plt.title('Electricity Demand Forecast for the First 2 Weeks of January 2026 — Model Comparison', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Demand (kWh)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.savefig('demand_forecast_january_2026_two_week_english.png')
plt.show()

#### Metricas

In [ ]:
metrics_jan_2026 = []

# Alinear los dataframes por índice para calcular métricas
common_index = df_pred_all_models.index.intersection(df_demand_real_january.index)

y_true_jan = df_demand_real_january.loc[common_index, 'demand_kwh']

for col in df_pred_all_models.columns:
    model_name = col.replace('pred_', '')
    y_pred_jan = df_pred_all_models.loc[common_index, col].fillna(0)

    valid_indices = y_pred_jan.dropna().index
    if len(valid_indices) > 0:
        metrics = compute_metrics(y_true_jan.loc[valid_indices], y_pred_jan.loc[valid_indices])
        metrics_jan_2026.append({
            'Modelo': model_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
            'R2': metrics['R2']
        })
    else:
        metrics_jan_2026.append({
            'Modelo': model_name,
            'MAE': np.nan,
            'RMSE': np.nan,
            'MAPE': np.nan,
            'R2': np.nan
        })

df_metrics_jan_2026 = pd.DataFrame(metrics_jan_2026).sort_values('MAPE')

print("\n" + "="*80)
print("  MÉTRICAS DE ERROR PARA ENERO 2026 — COMPARACIÓN DE MODELOS")
print("="*80)
print(df_metrics_jan_2026.to_string(index=False, float_format='{:.4f}'.format))

df_metrics_jan_2026.to_csv('metrics_jan_2026.csv', index=False)

In [ ]:
metrics_jan_2026 = []

# Define the first week
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-07 23:00:00')

# Filter predictions for the first week
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Filter real demand for the first week
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Alinear los dataframes por índice para calcular métricas
common_index = df_pred_week1.index.intersection(df_demand_real_week1.index)

y_true_jan = df_demand_real_week1.loc[common_index, 'demand_kwh']

for col in df_pred_week1.columns:
    model_name = col.replace('pred_', '')
    y_pred_jan = df_pred_week1.loc[common_index, col].fillna(0)

    valid_indices = y_pred_jan.dropna().index
    if len(valid_indices) > 0:
        metrics = compute_metrics(y_true_jan.loc[valid_indices], y_pred_jan.loc[valid_indices])
        metrics_jan_2026.append({
            'Modelo': model_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
            'R2': metrics['R2']
        })
    else:
        metrics_jan_2026.append({
            'Modelo': model_name,
            'MAE': np.nan,
            'RMSE': np.nan,
            'MAPE': np.nan,
            'R2': np.nan
        })

df_metrics_jan_2026 = pd.DataFrame(metrics_jan_2026).sort_values('MAPE')

print("\n" + "="*80)
print("  MÉTRICAS DE ERROR PARA ENERO 2026 — COMPARACIÓN DE MODELOS (Primera Semana)")
print("="*80)
print(df_metrics_jan_2026.to_string(index=False, float_format='{:.4f}'.format))


df_metrics_jan_2026.to_csv('metrics_jan_2026_one_week.csv', index=False)

In [ ]:
metrics_jan_2026 = []

# Define the first week
start_date_week1 = pd.Timestamp('2026-01-01 00:00:00')
end_date_week1 = pd.Timestamp('2026-01-14 23:00:00')

# Filter predictions for the first week
df_pred_week1 = df_pred_all_models.loc[start_date_week1:end_date_week1]

# Filter real demand for the first week
df_demand_real_week1 = df_demand_real_january.loc[start_date_week1:end_date_week1]

# Alinear los dataframes por índice para calcular métricas
common_index = df_pred_week1.index.intersection(df_demand_real_week1.index)

y_true_jan = df_demand_real_week1.loc[common_index, 'demand_kwh']

for col in df_pred_week1.columns:
    model_name = col.replace('pred_', '')
    y_pred_jan = df_pred_week1.loc[common_index, col].fillna(0)

    valid_indices = y_pred_jan.dropna().index
    if len(valid_indices) > 0:
        metrics = compute_metrics(y_true_jan.loc[valid_indices], y_pred_jan.loc[valid_indices])
        metrics_jan_2026.append({
            'Modelo': model_name,
            'MAE': metrics['MAE'],
            'RMSE': metrics['RMSE'],
            'MAPE': metrics['MAPE'],
            'R2': metrics['R2']
        })
    else:
        metrics_jan_2026.append({
            'Modelo': model_name,
            'MAE': np.nan,
            'RMSE': np.nan,
            'MAPE': np.nan,
            'R2': np.nan
        })

df_metrics_jan_2026 = pd.DataFrame(metrics_jan_2026).sort_values('MAPE')

print("\n" + "="*80)
print("  MÉTRICAS DE ERROR PARA ENERO 2026 — COMPARACIÓN DE MODELOS (Primeras 2 Semanas)")
print("="*80)
print(df_metrics_jan_2026.to_string(index=False, float_format='{:.4f}'.format))

df_metrics_jan_2026.to_csv('metrics_jan_2026_two_week.csv', index=False)

---
## 10. Resumen Ejecutivo y Exportación de Resultados

In [ ]:
# ============================================================
# 10. RESUMEN FINAL
# ============================================================
print("\n" + "="*100)
print("  RESUMEN EJECUTIVO — BENCHMARK COMPARATIVO")
print("="*100)

# Mejor modelo global
best_model = summary_df.iloc[0]
print(f"\n  🏆 MEJOR MODELO: {best_model['Modelo']}")
print(f"     Familia:  {best_model['Familia']}")
print(f"     MAPE:     {best_model['MAPE (%)']:.4f}%")
print(f"     RMSE:     {best_model['RMSE']:.2f}")
print(f"     MAE:      {best_model['MAE']:.2f}")
print(f"     R²:       {best_model['R²']:.4f}")

# Mejor por familia
print("\n  Mejor por familia:")
for family in ['MoE', 'Estadístico', 'ML', 'Deep Learning']:
    fam_df = summary_df[summary_df['Familia'] == family]
    if len(fam_df) > 0:
        best = fam_df.iloc[0]
        print(f"    {family:15s}: {best['Modelo']:25s} — MAPE: {best['MAPE (%)']:.4f}%")

# Significancia estadística
print(f"\n  Pruebas Estadísticas:")
print(f"    ANOVA p-value:   {p_value_anova:.2e} {'✓' if p_value_anova < 0.05 else '✗'}")
try:
    print(f"    Friedman p-value: {p_value_friedman:.2e} {'✓' if p_value_friedman < 0.05 else '✗'}")
except:
    pass

# Guardar todo
results_export = {
    'summary': summary_df.to_dict('records'),
    'anova_f_stat': float(f_stat),
    'anova_p_value': float(p_value_anova),
    'best_model': best_model['Modelo'],
    'timestamp': datetime.now().isoformat(),
    'n_random_search_iter': N_RANDOM_SEARCH_ITER,
    'methodology': 'Bergstra & Bengio (2012) Random Search',
}

with open(BENCH_DIR / 'benchmark_results.json', 'w') as f:
    json.dump(results_export, f, indent=2, default=str)

print(f"\n  Archivos generados en {BENCH_DIR}:")
for f in sorted(BENCH_DIR.iterdir()):
    print(f"    ✓ {f.name}")

print("\n" + "="*100)
print("  ✅ BENCHMARK COMPLETO")
print("="*100)